# Depth Anything V2 Small — DIMER E2E supervised adaptation tutorial: zero-shot vs neck/head vs bounded unfreeze on DIODE (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/depth-anything-depth-estimation-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/depth-anything-depth-estimation-pipeline/blob/main/tutorials/depth_anything_depth_estimation_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-depth--anything%2FDepth--Anything--V2--Small--hf-ffcc4d?style=flat)](https://huggingface.co/depth-anything/Depth-Anything-V2-Small-hf) [![Upstream](https://img.shields.io/badge/Upstream-DepthAnything%2FDepth--Anything--V2-181717?style=flat&logo=github&logoColor=white)](https://github.com/DepthAnything/Depth-Anything-V2) [![arXiv](https://img.shields.io/badge/arXiv-2406.09414-b31b1b.svg)](https://arxiv.org/abs/2406.09414)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** monocular relative depth estimation from one RGB still image and bounded supervised adaptation — the DPT neck and head on the frozen backbone with an optional unfreeze of the last transformer blocks, selected on validation against the untouched checkpoint — measured by affine-aligned AbsRel and δ1 against metric depth, using the pinned Depth Anything V2 Small weights

**This notebook is standalone.** It carries the repository's package (3 modules under `src/depth_anything_depth_estimation_pipeline/`, at revision `da877a1f8234`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `5426e4f0f36572d16453bbda7a8389317b1bef99` (~99 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned Depth Anything V2 Small snapshot (safetensors, 99 MB), fetches 40 digest-pinned DIODE validation views — RGB, metric depth and validity mask, 312 MB in total — from the Hugging Face Hub mirror (no credential), validates them and draws 24 / 8 / 8 training, validation and test views by a seeded split of whole scans, runs one test view through the inference contract with an input manifest, a rejection probe and a `sample-sanity` evaluation report against its metric reference, scores the untouched checkpoint on the test split beside a constant prior and a vertical-gradient prior (the **zero-shot policy**), trains the DPT neck and head on the frozen backbone (the **frozen policy**) and then the last two transformer blocks with them (the **unfrozen policy**), selects the policy ladder's best epoch by validation AbsRel — where epoch 0 is the untouched checkpoint — scores the held-out split with the selected model, renders depth maps before and after, exports the trained tensors as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On CPU the whole path takes about four minutes of model time after the downloads; a CUDA runtime is used automatically when present.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to supply your own RGB-D records as a `.zip` holding `records.csv` (columns `id`, `image`, `depth`, `group`, optional `mask`; `group` — the scan, session or device — must be non-empty on every row, or the loader refuses the set) beside the image files and `.npy` arrays (depth in metres, mask boolean) — files are decoded from the archive, never extracted to disk. They pass through the same validation, seeded group-disjoint split, priors, zero-shot scoring, policy ladder, held-out evaluation, preview, artifact export and reload-parity cells as the DIODE sample. The expected schema and the ceilings are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

At inference the image is rescaled so its sides are multiples of 14 near 518 px, passed through the DINOv2 ViT-S/14 backbone and the DPT neck and head, and the raw prediction is interpolated back to the input resolution: a per-pixel map of **relative inverse depth** — larger is nearer, the scale and shift are unknown, and the values are **not metres**. The carried pipeline module adds manifest verification, input validation with named ceilings, the resize back to the input resolution, and the `abs_rel`, `validate_inputs` and `evaluation_report` helpers; `evaluation_report` becomes `sample-sanity` only when a caller supplies metric reference depth — which this notebook, unlike its inference-only predecessor, does.

What this notebook adds to inference is **supervised adaptation under an explicit policy ladder**. The dataset is real and carries metric ground truth: 40 views from the DIODE validation release (CC BY 4.0) — the first two views of each of its 20 laser-scanned scenes, 10 indoor and 10 outdoor — pinned per file by byte size and SHA-256 as served by the Marigold evaluation mirror on the Hugging Face Hub at an immutable commit, fetched at run time and refused on any mismatch. Views of one scan share a scene, so the sample is split by **scan**, never by view. The carried `metrics.py` scores a prediction by **AbsRel** and **δ1** after a per-image least-squares alignment of the prediction to inverse reference depth (the standard protocol for a relative-depth model, with the far cap at 350 m); a **constant prior** (the image's own median depth) and a **vertical-gradient prior** (nearer at the bottom) frame the numbers. The **zero-shot policy** is the checkpoint as published; the **frozen policy** trains the DPT neck and head (2.7 M parameters) on the frozen backbone under a scale-and-shift-invariant loss; the **unfrozen policy** continues by training the last two transformer blocks with them; the epoch with the lowest validation AbsRel is kept — epoch 0, the untouched checkpoint, competes on equal terms, so the honest outcome "do not adapt" is available. The adaptation question is whether 24 DIODE training views buy anything over a model trained on 62 million images. Nothing here is a quality claim about your images: it is one seeded split of one small corpus.

**Learning objectives:** install the pinned runtime; read what the carried pipeline, metrics and dataset modules guarantee; stage and digest-verify the immutable upstream snapshot; fetch a digest-pinned real RGB-D corpus and validate and split it by scan without leakage; run one view through the public inference API and read a `sample-sanity` evaluation report against metric depth; read AbsRel and δ1 beside two trivial priors and understand why a relative-depth prediction must be aligned before it is scored; run a three-policy adaptation ladder with explicit hyperparameters and validation-based selection in which the untouched checkpoint can win; evaluate on an independent scan-disjoint test split; compare depth maps before and after; and export a safetensors adapter (neck and head plus any trained blocks) that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** metric (absolute) depth, video or temporal depth, stereo or multi-view fusion, surface normals, 3D reconstruction, batch inference, data augmentation, full-backbone or patch-embedding training, and any claim that 20 DIODE scenes stand in for your images. The repository exposes none of these; an adapted model still emits relative inverse depth.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available; float32 on both. The build record measured about 0.4 s per 1024 × 768 view to predict on CPU (3.2 s for the 8-view test split) and about 0.8 s per view per training step on the last two blocks with the neck and head, so the five-epoch ladder over 24 views with six validation passes took about 150 s. The pinned `torch==2.14.0` install and the 99 MB checkpoint are the large downloads of the run, then the 312 MB of DIODE files.
- **Knowledge:** basic Python and NumPy; the difference between inverse depth (larger = nearer, arbitrary scale and shift) and metric distance; why a least-squares alignment is needed before a relative prediction can be scored against metres; what AbsRel and δ1 measure; what validation-based selection between policies means.
- **Data contract:** records are `{{id, image, depth, mask}}` — a PIL image (or a path to one) with sides 14..4,096 px and aspect ratio at most 4.0, a float H × W array of metric depth in metres (or a path to a `.npy`), an optional boolean H × W validity mask (missing means `depth > 0`) covering at least 5 % of the pixels, depth at most 1,000 m, ids matching `[A-Za-z0-9_.:-]{{1,64}}` and unique; a training set needs 4..2,000 records; images are de-duplicated by decoded-pixel digest and split by `scan` / `group` so views of one scene never straddle splits. Only reference pixels inside 0.6..350 m are scored. BYOD accepts a `.zip` (or a directory) holding `records.csv` and the files, and requires a non-empty `group` on every row — the notebook's automatic split is group-disjoint only because the loader refuses ungrouped rows (`load_byod_dataset(..., require_group=False)` is the explicit opt-out, without that guarantee).
- **Validation is structural, not semantic:** nothing checks that a depth map belongs to its image or that its unit is metres beyond the 1,000 m ceiling — a mis-paired or mis-scaled set is trained on without complaint.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there — RGB-D captures of homes, workplaces or people are exactly that. The default path uploads nothing.
- **External access (data):** besides the Hub snapshot, the default path fetches 120 pinned objects — for each of 40 DIODE views the RGB PNG, the `_depth.npy` and the `_depth_mask.npy`, 312,448,846 bytes in total, one SHA-256 each in the carried `SAMPLE_RECORDS` table — from `huggingface.co/datasets/obukhovai/marigold_depth_eval` at commit `30c5b061` over HTTPS, each refused on any byte-size or SHA-256 mismatch before it is decoded; the files are DIODE's own (CC BY 4.0, Vasiljevic et al. 2019) as served by that mirror.
- **External access:** the Hugging Face Hub only, to fetch the pinned `depth-anything/Depth-Anything-V2-Small-hf` snapshot (~99 MB in total) at revision `5426e4f0f365…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'huggingface-hub==0.36.2',
]
NOTEBOOK_SOURCE = {
    'repository': 'depth-anything-depth-estimation-pipeline',
    'repository_revision': 'da877a1f8234ea126c768e7f099815edf2fddeae',
    'embedded_module': 'src/depth_anything_depth_estimation_pipeline/pipeline.py',
    'embedded_modules': ['src/depth_anything_depth_estimation_pipeline/metrics.py', 'src/depth_anything_depth_estimation_pipeline/pipeline.py', 'src/depth_anything_depth_estimation_pipeline/samples.py'],
    'module_sha256': 'f998096a65853d94cc261efa24f0a3d0f2c2749c65ee88c1a9769b1b383c006c',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/depth_anything_depth_estimation_pipeline/` @ `da877a1f8234`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/depth_anything_depth_estimation_pipeline/metrics.py`

In [ ]:
"""Depth metrics for the adaptation contract, implemented here with no external scorer.

The model emits **relative inverse depth** (disparity up to an unknown per-image scale and shift), so every
prediction is first aligned to the reference by least squares in inverse-depth space over the valid pixels
(``1 / depth = s * pred + t``), inverted to metres, and then scored:

- **AbsRel** — ``mean(|est - ref| / ref)`` over valid pixels (lower is better; 0 is perfect);
- **δ1** — the fraction of valid pixels with ``max(est / ref, ref / est) < 1.25`` (higher is better;
  1 is perfect).

Only reference pixels inside ``EVAL_DEPTH_RANGE_M`` (0.6..350 m, the DIODE sensor range) are scored, and the
aligned inverse depth is floored at ``1 / 350 m`` so a pixel the fit pushes past the far cap counts as 350 m
rather than as an unbounded error — the standard cap of relative-depth evaluation. Both metrics are reported
per image and as the mean over images (each image weighs the same regardless of its valid pixel count), plus
per domain when records carry one. Two trivial priors frame the numbers: the **constant
prior** (every pixel at the image's own median reference depth — an oracle constant, the strongest flat
guess) and the **vertical-gradient prior** (inverse depth rising linearly from the top row to the bottom row,
aligned exactly like a model prediction — "the ground is nearer at the bottom of the frame").
"""

from __future__ import annotations

from collections.abc import Mapping, Sequence
from typing import Any

import numpy as np

DELTA_THRESHOLD = 1.25
MIN_VALID_PIXELS = 2
EVAL_DEPTH_RANGE_M = (0.6, 350.0)  # reference pixels scored; also the far cap of the aligned estimate

METRIC_DEFINITIONS = {
    "alignment": (
        "per image, least-squares scale and shift mapping the prediction onto 1 / reference depth over the "
        "valid pixels inside EVAL_DEPTH_RANGE_M; the aligned inverse depth is floored at 1 / the far cap"
    ),
    "abs_rel": "mean over valid pixels of |aligned estimate - reference| / reference, then mean over images",
    "delta1": (
        "fraction of valid pixels with max(estimate / reference, reference / estimate) < 1.25, "
        "then mean over images"
    ),
    "per_domain": "the same means restricted to the records of one domain (indoors / outdoor when present)",
}


def _valid(pred: np.ndarray, depth: np.ndarray, mask: np.ndarray | None) -> np.ndarray:
    pred = np.asarray(pred, dtype=np.float64)
    depth = np.asarray(depth, dtype=np.float64)
    if pred.shape != depth.shape:
        raise ValueError(f"shape mismatch: pred {pred.shape} vs depth {depth.shape}")
    low, high = EVAL_DEPTH_RANGE_M
    valid = np.isfinite(depth) & (depth >= low) & (depth <= high) & np.isfinite(pred)
    if mask is not None:
        mask = np.asarray(mask, dtype=bool)
        if mask.shape != depth.shape:
            raise ValueError(f"shape mismatch: mask {mask.shape} vs depth {depth.shape}")
        valid &= mask
    if int(valid.sum()) < MIN_VALID_PIXELS:
        raise ValueError(f"need at least {MIN_VALID_PIXELS} valid reference pixels")
    return valid


def align_inverse_depth(
    pred: np.ndarray, depth: np.ndarray, mask: np.ndarray | None = None
) -> tuple[np.ndarray, np.ndarray, dict[str, float]]:
    """Return (aligned metric estimate, reference) over valid pixels and the fitted scale / shift."""
    valid = _valid(pred, depth, mask)
    p = np.asarray(pred, dtype=np.float64)[valid]
    ref = np.asarray(depth, dtype=np.float64)[valid]
    target = 1.0 / ref
    design = np.stack([p, np.ones_like(p)], axis=1)
    (scale, shift), *_ = np.linalg.lstsq(design, target, rcond=None)
    estimate = 1.0 / np.clip(scale * p + shift, 1.0 / EVAL_DEPTH_RANGE_M[1], None)
    return estimate, ref, {"scale": float(scale), "shift": float(shift), "n_valid": int(valid.sum())}


def aligned_abs_rel(pred: np.ndarray, depth: np.ndarray, mask: np.ndarray | None = None) -> float:
    estimate, ref, _ = align_inverse_depth(pred, depth, mask)
    return float(np.mean(np.abs(estimate - ref) / ref))


def aligned_delta1(
    pred: np.ndarray, depth: np.ndarray, mask: np.ndarray | None = None, *, threshold: float = DELTA_THRESHOLD
) -> float:
    estimate, ref, _ = align_inverse_depth(pred, depth, mask)
    ratio = np.maximum(estimate / ref, ref / estimate)
    return float(np.mean(ratio < threshold))


def image_metrics(pred: np.ndarray, depth: np.ndarray, mask: np.ndarray | None = None) -> dict[str, Any]:
    estimate, ref, fit = align_inverse_depth(pred, depth, mask)
    ratio = np.maximum(estimate / ref, ref / estimate)
    return {
        "abs_rel": float(np.mean(np.abs(estimate - ref) / ref)),
        "delta1": float(np.mean(ratio < DELTA_THRESHOLD)),
        "n_valid": fit["n_valid"],
        "scale": fit["scale"],
        "shift": fit["shift"],
    }


def depth_metrics(preds: Sequence[np.ndarray], records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Aggregate per-image metrics for aligned predictions over `{id, depth, mask[, domain]}` records."""
    if len(preds) != len(records):
        raise ValueError(f"{len(preds)} predictions for {len(records)} records")
    if not records:
        raise ValueError("at least one record is required")
    per_image = []
    for pred, record in zip(preds, records, strict=True):
        row = image_metrics(pred, record["depth"], record.get("mask"))
        per_image.append({"id": record["id"], "domain": str(record.get("domain", "unspecified")), **row})
    domains = sorted({r["domain"] for r in per_image})
    per_domain = {
        d: {
            "n": sum(1 for r in per_image if r["domain"] == d),
            "abs_rel": float(np.mean([r["abs_rel"] for r in per_image if r["domain"] == d])),
            "delta1": float(np.mean([r["delta1"] for r in per_image if r["domain"] == d])),
        }
        for d in domains
    }
    return {
        "n": len(per_image),
        "abs_rel": float(np.mean([r["abs_rel"] for r in per_image])),
        "delta1": float(np.mean([r["delta1"] for r in per_image])),
        "per_domain": per_domain,
        "per_image": per_image,
        "definitions": dict(METRIC_DEFINITIONS),
    }


def constant_prior(record: Mapping[str, Any]) -> dict[str, Any]:
    """The oracle constant: every pixel at the image's own median reference depth (no alignment needed)."""
    depth = np.asarray(record["depth"], dtype=np.float64)
    valid = _valid(depth, depth, record.get("mask"))
    ref = depth[valid]
    constant = float(np.median(ref))
    ratio = np.maximum(constant / ref, ref / constant)
    return {
        "abs_rel": float(np.mean(np.abs(constant - ref) / ref)),
        "delta1": float(np.mean(ratio < DELTA_THRESHOLD)),
        "n_valid": int(valid.sum()),
        "constant_m": constant,
    }


def vertical_gradient_prediction(height: int, width: int) -> np.ndarray:
    """Relative inverse depth rising from 0 at the top row to 1 at the bottom row (the ground-plane prior)."""
    rows = np.linspace(0.0, 1.0, num=height, dtype=np.float32)
    return np.repeat(rows[:, None], width, axis=1)


def prior_baselines(records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Both trivial priors scored on the same records as a model, each with the same aggregation."""
    constant_rows = []
    gradient_preds = []
    for record in records:
        constant_rows.append(
            {"id": record["id"], "domain": str(record.get("domain", "unspecified")), **constant_prior(record)}
        )
        h, w = np.asarray(record["depth"]).shape
        gradient_preds.append(vertical_gradient_prediction(h, w))
    gradient = depth_metrics(gradient_preds, records)
    return {
        "constant_prior": {
            "baseline": "every pixel at the image's own median reference depth (oracle constant)",
            "n": len(constant_rows),
            "abs_rel": float(np.mean([r["abs_rel"] for r in constant_rows])),
            "delta1": float(np.mean([r["delta1"] for r in constant_rows])),
            "per_image": constant_rows,
        },
        "vertical_gradient_prior": {
            "baseline": (
                "inverse depth rising linearly from the top row to the bottom row, "
                "affine-aligned like a prediction"
            ),
            **{k: gradient[k] for k in ("n", "abs_rel", "delta1", "per_domain", "per_image")},
        },
    }

**Module 2/3:** `src/depth_anything_depth_estimation_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
"""Monocular relative depth estimation over the pinned Depth Anything V2 Small checkpoint, plus a bounded
supervised-adaptation contract.

Inference (`predict`) is unchanged: one PIL image in, relative inverse depth at the input resolution out.
The adaptation contract (`evaluate`, `adapt`, `save_artifact`, `from_artifact`) scores the model on a
validated `{id, image, depth, mask}` dataset by affine-aligned AbsRel and δ1 (`metrics.py`), fine-tunes the
DPT neck and head on the frozen backbone (the **frozen policy**) and optionally the last transformer blocks
with them (the **unfrozen policy**) under a scale-and-shift-invariant loss, keeps the epoch with the lowest
validation AbsRel — where epoch 0 is the untouched checkpoint, the **zero-shot policy**, competing on equal
terms — and exports the trained tensors as a safetensors adapter bound to the pinned base weights.
"""

from __future__ import annotations

import hashlib
import json
import math
import random
import time
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image

MODEL_ID = "depth-anything/Depth-Anything-V2-Small-hf"
MODEL_REVISION = "5426e4f0f36572d16453bbda7a8389317b1bef99"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "depth-anything-v2-small"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Input ceilings. The DPT processor rescales the image so that its sides are multiples of 14 close
# to 518 px, and the raw prediction is interpolated back to the caller's resolution, so the cost that
# grows with the caller's image is the aspect ratio (backbone tokens) and the output interpolation.
MAX_IMAGE_SIDE = 4096
MIN_IMAGE_SIDE = 14
MAX_ASPECT_RATIO = 4.0
DEPTH_KIND = "relative"
WEIGHTS_FILE = "model.safetensors"
WEIGHT_SHA256 = (
    "3152477ce0d8d6978d76b995120de97cb5b928701fd0f817769f59e249a16b70"  # manifest digest of WEIGHTS_FILE
)
PARAMETER_COUNT = 24_785_089  # backbone 22,056,576 + neck 2,700,768 + head 27,745
TRANSFORMER_BLOCKS = 12  # DINOv2 ViT-S/14 backbone depth
DEFAULT_TRAINABLE_BLOCKS = (
    2  # the unfrozen policy trains the last two blocks (3,550,464 parameters) with neck + head
)
NECK_HEAD_PARAMETERS = 2_728_513  # the DPT neck and head, trained under every adapted policy
ADAPTER_PREFIXES = ("neck.", "head.")
BLOCK_PREFIX = "backbone.encoder.layer."
POLICY_ZERO_SHOT = "zero-shot (no adaptation)"
POLICY_FROZEN = "frozen backbone + DPT neck and head"
POLICY_UNFROZEN = "unfrozen last {k} blocks + DPT neck and head"
MIN_VALID_TARGET_PIXELS = 2  # the affine alignment inside the loss needs at least two valid pixels
MAX_EVAL_RECORDS = 2_000
MIN_SCORED_RECORDS = 20  # below this a scored dataset is labelled a small sample
ARTIFACT_FORMAT = "org.valcorza.depth-anything-v2-small.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {
        "path": str(root),
        "model_id": manifest["modelId"],
        "revision": manifest["revision"],
        "files": len(manifest["files"]),
        "total_bytes": manifest.get("totalBytes"),
    }


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def abs_rel(pred: np.ndarray, ref_depth: np.ndarray, *, align: bool = True) -> float:
    """Absolute relative error of a relative inverse-depth map against caller-supplied metric depth.

    The model emits relative inverse depth (disparity up to an unknown scale and shift), so the
    prediction is first aligned to ``1 / ref_depth`` by least squares over valid pixels
    (``ref_depth`` inside ``metrics.EVAL_DEPTH_RANGE_M``, 0.6..350 m), inverted to depth with the
    aligned inverse depth floored at ``1 / 350 m`` (the far cap of relative-depth evaluation, so a pixel
    pushed past it counts as 350 m rather than as an unbounded error), and scored as
    ``mean(|est - ref| / ref)``. With ``align=False`` the arrays are compared as given, which is only
    meaningful for metric input. `metrics.aligned_abs_rel` is the same computation with a mask argument.
    """
    pass  # standalone rewrite (build_notebook.py): `from .metrics import EVAL_DEPTH_RANGE_M` removed — names are kernel globals defined by the carried modules

    pred = np.asarray(pred, dtype=np.float64)
    ref_depth = np.asarray(ref_depth, dtype=np.float64)
    if pred.shape != ref_depth.shape:
        raise ValueError(f"shape mismatch: pred {pred.shape} vs ref {ref_depth.shape}")
    low, high = EVAL_DEPTH_RANGE_M
    valid = np.isfinite(ref_depth) & (ref_depth >= low) & (ref_depth <= high) & np.isfinite(pred)
    if valid.sum() < 2:
        raise ValueError(f"need at least 2 valid reference pixels ({low} <= ref_depth <= {high})")
    if align:
        target = 1.0 / ref_depth[valid]
        design = np.stack([pred[valid], np.ones(int(valid.sum()))], axis=1)
        (scale, shift), *_ = np.linalg.lstsq(design, target, rcond=None)
        estimate = 1.0 / np.clip(scale * pred[valid] + shift, 1.0 / high, None)
    else:
        estimate = pred[valid]
    return float(np.mean(np.abs(estimate - ref_depth[valid]) / ref_depth[valid]))


def validate_image(image: Any) -> Image.Image:
    """Type- and size-check a caller image and return it as RGB."""
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    short, long = min(width, height), max(width, height)
    if short < MIN_IMAGE_SIDE:
        raise ValueError(f"image side {short} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")
    if long > MAX_IMAGE_SIDE:
        raise ValueError(f"image side {long} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")
    if long / short > MAX_ASPECT_RATIO:
        raise ValueError(f"aspect ratio {long / short:.2f} > MAX_ASPECT_RATIO {MAX_ASPECT_RATIO}")
    return image.convert("RGB")


INPUT_SCHEMA: dict[str, Any] = {
    "input": "PIL.Image.Image, or a sequence of them for the validation stage; any mode, converted to RGB",
    "short_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "long_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "aspect_ratio": [1.0, MAX_ASPECT_RATIO],
    "output": f"{DEPTH_KIND} inverse depth, float32 H x W at the input resolution (larger = nearer)",
    "preprocessing": (
        "convert to RGB; the DPT processor rescales the sides to multiples of 14 near 518 px and the "
        "raw prediction is interpolated back (bicubic) to the input resolution"
    ),
}


def validate_inputs(images: Any, *, names: Sequence[str] | None = None) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, per-input observations, verdict).

    Each image is routed through the public ``validate_image`` that ``predict`` itself calls, so a
    rejection here raises exactly what ``predict`` would; a caller that wants the finding recorded
    catches the exception and stores ``str(exc)`` under ``findings``.
    """
    batch = [images] if isinstance(images, Image.Image) else images
    if not isinstance(batch, Sequence) or isinstance(batch, str | bytes):
        raise TypeError("images must be a PIL.Image.Image or a sequence of them")
    if len(batch) < 1:
        raise ValueError("at least one image is required")
    if names is not None and len(names) != len(batch):
        raise ValueError("names must have one entry per image")
    inputs = []
    for index, candidate in enumerate(batch):
        rgb = validate_image(candidate)
        width, height = rgb.size
        long_side, short_side = max(width, height), min(width, height)
        inputs.append(
            {
                "id": names[index] if names else f"image-{index}",
                "mode": getattr(candidate, "mode", rgb.mode),
                "size": [width, height],
                "aspect_ratio": round(long_side / short_side, 3),
            }
        )
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": inputs,
        "n_images": len(inputs),
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any],
    reference_depth: Any | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``reference_depth`` (metric depth in metres, same H x W as the prediction) the report carries
    ``abs_rel`` computed by the repository's own helper after least-squares affine alignment in inverse
    depth, as sample-sanity evidence. Without it the verdict is ``not-measurable`` and the report says
    what ground truth would make the task measurable: relative inverse depth has no intrinsic score.
    """
    depth = np.asarray(result["depth"])
    base = {
        "task": "monocular relative depth estimation",
        "score_semantics": (
            f"{result.get('depth_kind', DEPTH_KIND)} inverse depth with unknown per-image scale and shift: "
            "larger is nearer, values are not metres, carry no confidence, and no threshold is shipped"
        ),
        "sample_kind": sample_kind,
        "n_images": 1,
        "n_pixels": int(depth.size),
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if reference_depth is None:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no metric reference depth was supplied for the evaluated image",
            "needs": (
                "a metric depth map in metres with the same height and width as the image, from a depth "
                "sensor, LiDAR, or an RGB-D benchmark, scored with abs_rel(pred, ref_depth, align=True) "
                "against a constant-depth or vertical-gradient prior as the trivial baseline"
            ),
        }
    pass  # standalone rewrite (build_notebook.py): `from .metrics import EVAL_DEPTH_RANGE_M` removed — names are kernel globals defined by the carried modules

    ref = np.asarray(reference_depth, dtype=np.float64)
    valid = int((np.isfinite(ref) & (ref >= EVAL_DEPTH_RANGE_M[0]) & (ref <= EVAL_DEPTH_RANGE_M[1])).sum())
    return {
        **base,
        "metrics": [
            {
                "id": "abs_rel",
                "value": abs_rel(depth, ref, align=True),
                "align": True,
                "n_valid_pixels": valid,
                "estimation": (
                    "single image, least-squares affine alignment in inverse depth, no dispersion estimate"
                ),
            }
        ],
        "verdict": "sample-sanity",
        "reason": "one image with caller-supplied metric depth from the tutorial sample; not a benchmark",
        "needs": "a held-out set of metric depth maps from the deployment domain for any generalisable claim",
    }


@dataclass
class DepthAnythingPipeline:
    """Monocular relative depth estimation over the pinned Depth Anything V2 Small checkpoint."""

    _runner: Callable[[Image.Image], np.ndarray]
    device: str
    source: str = "injected"
    _model: Any = field(default=None, repr=False)
    _processor: Any = field(default=None, repr=False)
    adapter: dict[str, Any] | None = field(default=None, repr=False)

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> DepthAnythingPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            source, kwargs, origin = str(root), {"local_files_only": True}, "local-snapshot"
        elif allow_download:
            source, kwargs, origin = MODEL_ID, {}, "hub"
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage {MODEL_ID}@{MODEL_REVISION} under weights/{MODEL_KEY}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import AutoImageProcessor, AutoModelForDepthEstimation

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        processor = AutoImageProcessor.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs
        )
        model = AutoModelForDepthEstimation.from_pretrained(
            source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs
        )
        model = model.to(resolved_device).eval()

        def runner(image: Image.Image) -> np.ndarray:
            inputs = processor(images=image, return_tensors="pt").to(resolved_device)
            with torch.inference_mode():
                predicted = model(**inputs).predicted_depth
            resized = torch.nn.functional.interpolate(
                predicted.unsqueeze(1), size=image.size[::-1], mode="bicubic", align_corners=False
            )
            return resized[0, 0].float().cpu().numpy()

        return cls(runner, resolved_device, origin, _model=model, _processor=processor)

    def predict(self, image: Image.Image) -> dict[str, Any]:
        """Return relative inverse depth as a float32 H x W array at the input resolution."""
        rgb = validate_image(image)
        depth = np.asarray(self._runner(rgb), dtype=np.float32)
        if depth.shape != (rgb.height, rgb.width):
            raise RuntimeError(f"backend returned shape {depth.shape}, expected {(rgb.height, rgb.width)}")
        return {
            "depth": depth,
            "depth_kind": DEPTH_KIND,
            "depth_min": float(depth.min()),
            "depth_max": float(depth.max()),
            "height": rgb.height,
            "width": rgb.width,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    # ---- adaptation ----
    def _require_model(self) -> tuple[Any, Any]:
        if self._model is None or self._processor is None:
            raise ValueError(
                "this operation needs a pipeline built with from_pretrained() or from_artifact()"
            )
        return self._model, self._processor

    def _trainable_names(self, trainable_blocks: int) -> list[str]:
        """`neck.*` and `head.*` always; plus the last `trainable_blocks` backbone blocks."""
        if isinstance(trainable_blocks, bool) or not isinstance(trainable_blocks, int):
            raise ValueError(f"trainable_blocks must be an int in 0..{TRANSFORMER_BLOCKS}")
        if not 0 <= trainable_blocks <= TRANSFORMER_BLOCKS:
            raise ValueError(f"trainable_blocks must be an int in 0..{TRANSFORMER_BLOCKS}")
        model, _ = self._require_model()
        first = TRANSFORMER_BLOCKS - trainable_blocks
        names = []
        for name, _p in model.named_parameters():
            is_block = name.startswith(BLOCK_PREFIX) and int(name[len(BLOCK_PREFIX) :].split(".")[0]) >= first
            if name.startswith(ADAPTER_PREFIXES) or is_block:
                names.append(name)
        return names

    def evaluate(self, records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
        """Score `predict` on validated `{id, image, depth, mask}` records: affine-aligned AbsRel and δ1."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import depth_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        started = time.perf_counter()
        preds = [self.predict(r["image"])["depth"] for r in checked]
        metrics = depth_metrics(preds, checked)
        return {
            **metrics,
            "policy": self.adapter["policy"] if self.adapter else POLICY_ZERO_SHOT,
            "adapted": self.adapter is not None,
            "verdict": "measured" if len(checked) >= MIN_SCORED_RECORDS else "measured-small-sample",
            "seconds": round(time.perf_counter() - started, 3),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    @staticmethod
    def _targets(record: Mapping[str, Any], size: tuple[int, int], device: str) -> tuple[Any, Any]:
        """Inverse-depth target and validity mask at the model's working resolution (nearest resampling)."""
        import torch

        depth = torch.from_numpy(np.ascontiguousarray(record["depth"], dtype=np.float32))[None, None]
        mask = torch.from_numpy(np.ascontiguousarray(record["mask"], dtype=np.float32))[None, None]
        depth = torch.nn.functional.interpolate(depth, size=size, mode="nearest")[0, 0]
        valid = torch.nn.functional.interpolate(mask, size=size, mode="nearest")[0, 0] > 0.5
        valid &= depth > 0
        if int(valid.sum()) < MIN_VALID_TARGET_PIXELS:
            raise ValueError(
                f"record {record.get('id', '?')!r} keeps {int(valid.sum())} valid pixels at the working "
                f"resolution; at least {MIN_VALID_TARGET_PIXELS} are needed for the affine alignment"
            )
        return (1.0 / depth.clamp_min(1e-6)).to(device), valid.to(device)

    @staticmethod
    def _ssi_loss(pred: Any, target: Any, valid: Any) -> Any:
        """Scale-and-shift-invariant squared error: the prediction is affinely aligned to the inverse-depth
        target over valid pixels in closed form (gradients flow through the fit), residuals normalised by
        the target's mean magnitude so images at different depths weigh alike."""
        p = pred[valid]
        t = target[valid]
        n = float(p.numel())
        sp, st, spp, spt = p.sum(), t.sum(), (p * p).sum(), (p * t).sum()
        det = (n * spp - sp * sp).clamp_min(1e-12)
        scale = (n * spt - sp * st) / det
        shift = (st - scale * sp) / n
        residual = (scale * p + shift - t) / (t.abs().mean() + 1e-6)
        return (residual * residual).mean()

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        trainable_blocks: int = DEFAULT_TRAINABLE_BLOCKS,
        head_epochs: int = 2,
        epochs: int = 3,
        lr: float = 1e-5,
        seed: int = 0,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded supervised adaptation under an explicit policy ladder, selected on validation.

        Epoch 0 is the untouched checkpoint (the **zero-shot policy**). Stage A (the **frozen policy**) trains
        the DPT neck and head on the frozen backbone for `head_epochs` epochs. Stage B (the **unfrozen
        policy**, when `trainable_blocks` > 0 and `epochs` > 0) continues with the last `trainable_blocks`
        transformer blocks unfrozen for `epochs` more epochs. Every epoch trains one image at a time with
        AdamW (`lr`, weight decay 0.01, gradient clipping 1.0, seeded order, no augmentation) under the
        scale-and-shift-invariant loss and is scored on `val` by `evaluate`; the epoch with the **lowest
        validation AbsRel** is kept and its tensors restored — so the outcome can be "do not adapt".
        Without `val` the final epoch is kept.
        """
        if isinstance(head_epochs, bool) or not isinstance(head_epochs, int) or not 0 <= head_epochs <= 20:
            raise ValueError("head_epochs must be an int in 0..20")
        if isinstance(epochs, bool) or not isinstance(epochs, int) or not 0 <= epochs <= 20:
            raise ValueError("epochs must be an int in 0..20")
        if not isinstance(lr, int | float) or not 0.0 < float(lr) <= 1e-3:
            raise ValueError("lr must be in (0, 1e-3]")
        if isinstance(trainable_blocks, bool) or not isinstance(trainable_blocks, int):
            raise ValueError(f"trainable_blocks must be an int in 0..{TRANSFORMER_BLOCKS}")
        if not 0 <= trainable_blocks <= TRANSFORMER_BLOCKS:
            raise ValueError(f"trainable_blocks must be an int in 0..{TRANSFORMER_BLOCKS}")
        model, processor = self._require_model()
        import torch

        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        names_b = self._trainable_names(trainable_blocks)
        names_a = [n for n in names_b if n.startswith(ADAPTER_PREFIXES)]
        train_records = validate_dataset(train)["records"]
        val_records = (
            validate_dataset(val, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
            if val is not None
            else None
        )
        params = dict(model.named_parameters())
        started = time.perf_counter()

        def brief(metrics: Mapping[str, Any] | None) -> dict[str, float] | None:
            if metrics is None:
                return None
            return {
                "n": metrics["n"],
                "abs_rel": round(metrics["abs_rel"], 6),
                "delta1": round(metrics["delta1"], 6),
            }

        def score() -> dict[str, float] | None:
            model.eval()
            return brief(self.evaluate(val_records)) if val_records is not None else None

        history: list[dict[str, Any]] = [
            {"epoch": 0, "stage": POLICY_ZERO_SHOT, "train_loss": None, "val": score()}
        ]
        best_epoch, best_score = 0, (history[0]["val"]["abs_rel"] if history[0]["val"] else math.inf)
        best_state = {n: params[n].detach().clone() for n in names_b}
        stages: list[tuple[str, list[str], int]] = [(POLICY_FROZEN, names_a, head_epochs)]
        if trainable_blocks > 0 and epochs > 0:
            stages.append((POLICY_UNFROZEN.format(k=trainable_blocks), names_b, epochs))
        torch.manual_seed(seed)
        rng = random.Random(seed)
        epoch = 0
        initial_state = {n: v.clone() for n, v in best_state.items()}
        try:
            for stage, names, n_epochs in stages:
                if n_epochs == 0:
                    continue
                for p in model.parameters():
                    p.requires_grad_(False)
                for n in names:
                    params[n].requires_grad_(True)
                optimiser = torch.optim.AdamW([params[n] for n in names], lr=float(lr), weight_decay=0.01)
                for _ in range(n_epochs):
                    epoch += 1
                    model.train()
                    order = list(range(len(train_records)))
                    rng.shuffle(order)
                    losses = []
                    for index in order:
                        record = train_records[index]
                        pixel_values = processor(images=record["image"], return_tensors="pt")["pixel_values"]
                        pred = model(pixel_values=pixel_values.to(self.device)).predicted_depth[0]
                        target, valid = self._targets(record, tuple(pred.shape), self.device)
                        loss = self._ssi_loss(pred.float(), target, valid)
                        if not torch.isfinite(loss):
                            raise RuntimeError(
                                f"non-finite loss on record {record['id']!r}; adaptation aborted"
                            )
                        optimiser.zero_grad(set_to_none=True)
                        loss.backward()
                        torch.nn.utils.clip_grad_norm_([params[n] for n in names], 1.0)
                        optimiser.step()
                        losses.append(float(loss.detach()))
                    val_metrics = score()
                    entry = {
                        "epoch": epoch,
                        "stage": stage,
                        "train_loss": float(np.mean(losses)),
                        "val": val_metrics,
                    }
                    history.append(entry)
                    if progress is not None:
                        progress(entry)
                    current = val_metrics["abs_rel"] if val_metrics else -epoch  # no val: the last epoch wins
                    if current < best_score:
                        best_epoch, best_score = epoch, current
                        best_state = {n: params[n].detach().clone() for n in names_b}
        except BaseException:
            # Transactional: a failure in training, validation or the progress callback leaves the base
            # exactly as it was, frozen, with no adapter attached.
            with torch.no_grad():
                for n, value in initial_state.items():
                    params[n].copy_(value)
            for p in model.parameters():
                p.requires_grad_(False)
            model.eval()
            self.adapter = None
            raise
        with torch.no_grad():
            for n, value in best_state.items():
                params[n].copy_(value)
        for p in model.parameters():
            p.requires_grad_(False)
        model.eval()
        policy = history[best_epoch]["stage"]
        self.adapter = {
            "policy": policy,
            "trainable_blocks": trainable_blocks,
            "head_epochs": head_epochs,
            "epochs": epochs,
            "lr": float(lr),
            "seed": seed,
            "best_epoch": best_epoch,
            "selection": "lowest validation AbsRel (epoch 0 = zero-shot checkpoint)"
            if val_records is not None
            else "final epoch (no validation split)",
            "n_trainable_head": sum(params[n].numel() for n in names_a),
            "n_trainable_blocks": sum(
                params[n].numel() for n in names_b if not n.startswith(ADAPTER_PREFIXES)
            ),
            "n_total": sum(p.numel() for p in model.parameters()),
            "n_train": len(train_records),
            "n_val": len(val_records) if val_records is not None else 0,
            "history": history,
            "trainable_names": names_b if policy.startswith("unfrozen") else names_a,
            "seconds": round(time.perf_counter() - started, 3),
        }
        return dict(self.adapter)

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the DPT neck and head (and any trained block tensors) as safetensors with a manifest naming
        the base; the backbone blocks travel only when the unfrozen policy was selected."""
        if self.adapter is None:
            raise ValueError("nothing to save: call adapt() first")
        model, _ = self._require_model()
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adapter.get("trainable_names", []))
        tensors = {k: v.detach().cpu().contiguous() for k, v in model.state_dict().items() if k in names}
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {
                "id": MODEL_ID,
                "revision": MODEL_REVISION,
                "key": MODEL_KEY,
                "weight_file": WEIGHTS_FILE,
                "weight_sha256": WEIGHT_SHA256,
            },
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "no_op": self.adapter["policy"] == POLICY_ZERO_SHOT,
            "note": (
                "the zero-shot policy was selected: the neck and head tensors below are byte copies of the "
                "base and loading this artifact changes nothing"
                if self.adapter["policy"] == POLICY_ZERO_SHOT
                else "the neck and head tensors, plus the trained backbone blocks under the unfrozen policy"
            ),
            "history": self.adapter.get("history", []),
            "tensors": sorted(tensors),
            "files": [
                {
                    "path": ARTIFACT_WEIGHTS_NAME,
                    "bytes": weights_path.stat().st_size,
                    "sha256": _sha256(weights_path),
                }
            ],
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(
            json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
        )
        return out

    def _check_artifact_manifest(self, root: Path, manifest: Mapping[str, Any]) -> tuple[Path, str, int]:
        """Refuse an artifact whose manifest is not exactly the one this pipeline writes: the supported format
        and version, the pinned base (id, revision, weight file, digest), exactly one file entry named
        `adapter.safetensors` that resolves inside the artifact directory, a canonical policy and an integer
        `trainable_blocks` in range. Nothing is deserialised here. The digest check that follows detects
        corruption or drift of the weights relative to the adjacent manifest; it is not authenticity against
        an actor who can replace both files."""
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        if manifest.get("format_version") != ARTIFACT_FORMAT_VERSION:
            raise ValueError(
                f"artifact format_version {manifest.get('format_version')!r} is not the supported "
                f"{ARTIFACT_FORMAT_VERSION!r}"
            )
        base = manifest.get("base_model", {})
        if (base.get("id"), base.get("revision"), base.get("weight_sha256")) != (
            MODEL_ID,
            MODEL_REVISION,
            WEIGHT_SHA256,
        ):
            raise ValueError("artifact was adapted from a different base model, revision or weight file")
        if base.get("weight_file", WEIGHTS_FILE) != WEIGHTS_FILE:
            raise ValueError("artifact was adapted from a different base weight file")
        files = manifest.get("files")
        if not isinstance(files, list) or len(files) != 1:
            raise ValueError("artifact manifest must list exactly one file")
        entry = files[0]
        if not isinstance(entry, Mapping) or entry.get("path") != ARTIFACT_WEIGHTS_NAME:
            raise ValueError(f"artifact manifest must name exactly {ARTIFACT_WEIGHTS_NAME!r}")
        weights_path = (root / entry["path"]).resolve()
        if weights_path.parent != root.resolve():
            raise ValueError("artifact weight path must resolve inside the artifact directory")
        adapter = manifest.get("adapter")
        if not isinstance(adapter, Mapping):
            raise ValueError("artifact manifest has no adapter block")
        blocks = adapter.get("trainable_blocks")
        if isinstance(blocks, bool) or not isinstance(blocks, int) or not 0 <= blocks <= TRANSFORMER_BLOCKS:
            raise ValueError(
                f"artifact manifest does not record an integer trainable_blocks in 0..{TRANSFORMER_BLOCKS}"
            )
        policy = adapter.get("policy")
        if policy in (POLICY_ZERO_SHOT, POLICY_FROZEN):
            blocks = 0
        elif policy != POLICY_UNFROZEN.format(k=blocks) or blocks == 0:
            raise ValueError(
                f"artifact policy {policy!r} is not a canonical policy for trainable_blocks={blocks}"
            )
        if not isinstance(manifest.get("tensors"), list):
            raise ValueError("artifact manifest must list its tensors")
        return weights_path, policy, blocks

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Verify an adapter's manifest, digest and exact tensor set **before** deserialising, then overlay
        its tensors (a zero-shot artifact must equal the base and changes nothing)."""
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        weights_path, policy, blocks = self._check_artifact_manifest(root, manifest)
        entry = manifest["files"][0]
        if not weights_path.is_file():
            raise FileNotFoundError(f"artifact weights missing: {weights_path}")
        if _sha256(weights_path) != entry["sha256"] or weights_path.stat().st_size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: digest or size mismatch; refusing to load")
        # The exact tensor set the recorded policy implies: neck and head always, the last `blocks` blocks
        # only under the unfrozen policy.
        expected = sorted(self._trainable_names(blocks))
        if sorted(manifest["tensors"]) != expected:
            raise ValueError("artifact tensor list does not match its recorded policy and trainable_blocks")
        model, _ = self._require_model()
        import torch
        from safetensors.torch import load_file

        tensors = load_file(str(weights_path))
        if sorted(tensors) != expected:
            raise ValueError("artifact tensor names differ from its manifest")
        state = model.state_dict()
        if policy == POLICY_ZERO_SHOT and any(
            not torch.equal(value.to(state[key].dtype), state[key].cpu()) for key, value in tensors.items()
        ):
            raise ValueError("artifact claims the zero-shot policy but its tensors differ from the base")
        for key, value in tensors.items():
            if key not in state or not key.startswith((*ADAPTER_PREFIXES, BLOCK_PREFIX)):
                raise ValueError(
                    f"artifact tensor {key} is not an adaptable neck, head or transformer-block tensor"
                )
            if tuple(value.shape) != tuple(state[key].shape):
                raise ValueError(
                    f"artifact tensor {key}: shape {tuple(value.shape)} != {tuple(state[key].shape)}"
                )
        with torch.no_grad():
            params = dict(model.named_parameters())
            for key, value in tensors.items():
                params[key].copy_(value.to(params[key].dtype))
        model.eval()
        self.adapter = {
            **manifest["adapter"],
            "trainable_names": sorted(tensors),
            "history": manifest.get("history", []),
        }
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> DepthAnythingPipeline:
        pipeline = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipeline.load_artifact(artifact_dir)
        return pipeline

**Module 3/3:** `src/depth_anything_depth_estimation_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Depth-labelled image dataset contract for adapting the relative-depth estimator: the pinned DIODE
validation sample, validation, seeded scan-level splitting, BYOD loaders and CSV export.

The default dataset is **real** and carries metric ground truth: 40 views from the DIODE validation release
(Vasiljevic et al., 2019; CC BY 4.0) — the first two views in file-name order of every one of its 20 scans
(10 indoor, 10 outdoor) — chosen a priori on 2026-09-19 and pinned here per file (RGB PNG, depth `.npy` in
metres, validity-mask `.npy`) by byte size and SHA-256 as served by the Marigold evaluation mirror of DIODE on
the Hugging Face Hub at an immutable commit. Every file is fetched at run time and refused on any byte-size or
SHA-256 mismatch; the repository redistributes none of them. Views of one scan share a scene, so the sample is
split **by scan**, never by view.

A record is ``{id, image, depth, mask}``: a PIL image (or a path to one), a float32 H x W array of metric
depth in metres (or a path to a `.npy`) and a boolean H x W validity mask (or a path; missing means
``depth > 0``). ``domain`` / ``scene`` / ``scan`` are optional provenance keys; ``scan`` (or ``group``) is the
split unit when present.
"""

from __future__ import annotations

import csv
import hashlib
import io
import json
import random
import re
import urllib.request
import zipfile
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

import numpy as np
from PIL import Image

# standalone rewrite (build_notebook.py): `from .pipeline import MAX_ASPECT_RATIO, MAX_IMAGE_SIDE, MIN_IMAGE_SIDE, MODEL_ID` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "DIODE validation views (indoor + outdoor) with metric depth"
CORPUS_RELEASE = (
    "DIODE validation release as served by the Marigold evaluation mirror "
    "(obukhovai/marigold_depth_eval @ 30c5b061, Hugging Face Hub); 40 views pinned 2026-09-19"
)
CORPUS_COMMIT = "30c5b061d863e383e3ea9fa14555737a199d7ad9"
CORPUS_BASE_URL = (
    f"https://huggingface.co/datasets/obukhovai/marigold_depth_eval/resolve/{CORPUS_COMMIT}/diode/"
)
CORPUS_LICENSE = "CC BY 4.0 (DIODE: A Dense Indoor and Outdoor DEpth Dataset, Vasiljevic et al., 2019)"
CORPUS_BYTES = 312_448_846
DOMAINS = ("indoors", "outdoor")
FILE_SUFFIXES = (".png", "_depth.npy", "_depth_mask.npy")

# (record id, domain, scene, scan, file stem, then bytes + sha256 of the png, depth .npy and mask .npy)
SAMPLE_RECORDS: tuple[tuple[str, str, str, str, str, int, str, int, str, int, str], ...] = (
    (
        "view-000",
        "indoors",
        "scene_00019",
        "scan_00183",
        "00019_00183_indoors_000_010",
        1172425,
        "71f9713e01a4d9899184b63f293446e62953ee2e526d81e5da8f101f2400ae46",
        3145856,
        "a2bbbf55765ffec98a75d3a78f9925b28bfe476080d3e8eae86c645db4ac2b15",
        3145856,
        "e9b5f1cce7c4e8e03a94b006f81ef72d963bfb9b7187f8208f9e92067db823fa",
    ),
    (
        "view-001",
        "indoors",
        "scene_00019",
        "scan_00183",
        "00019_00183_indoors_000_040",
        1102623,
        "ba05f76208099f78dcf483908c7de07ad4d89784ae694aefdbb5a82dc2bfedf1",
        3145856,
        "c72d44e851b649ad787de4d0a21e5c115660779d4d1f8e054e8af201748c2c39",
        3145856,
        "304216b2ce8825e529d836e7ebe36dd80090c19b2e620029a158f1dd42a3e42c",
    ),
    (
        "view-002",
        "indoors",
        "scene_00020",
        "scan_00184",
        "00020_00184_indoors_050_000",
        1142437,
        "2641d27768b547e53d4d1dac922b72cb31a0413762bd78b36b898386c150d40e",
        3145856,
        "d4943f5e28555bfd59e609e7116539f2efd68c8a9378dcf696dca0e078f62004",
        3145856,
        "789dabf2482fa3e3d4bab67b81d49518e1b618f7ba55b82edae10ac8d18e4855",
    ),
    (
        "view-003",
        "indoors",
        "scene_00020",
        "scan_00184",
        "00020_00184_indoors_050_020",
        1017065,
        "a44432a3ba6bab0f6b4bb4326df5db4347a6eb577f74f83141c59aae6810b6a4",
        3145856,
        "d5f628673cfcbff12082cfe2dee75c71d8521560486946369dca90b369aa9b5c",
        3145856,
        "06077d743933e907c17b9e3e2b5146987e87f1b0a2122c784ada4f55a8f8b3f4",
    ),
    (
        "view-004",
        "indoors",
        "scene_00020",
        "scan_00185",
        "00020_00185_indoors_000_000",
        1208304,
        "1bf8317ae6d1ef6c412ff233bda2944568aa7e9169606ed5f694d9c2bc1032f9",
        3145856,
        "6de0b81413613d2abecea29e327220a6d94d54d5ab1bccca786a9720993a4e30",
        3145856,
        "afc7dad580ec7d8f41884032f0c97a8bcee31e76d22cc7c2ab988b28daef4dd0",
    ),
    (
        "view-005",
        "indoors",
        "scene_00020",
        "scan_00185",
        "00020_00185_indoors_000_020",
        1213267,
        "a632cec9d93174aaf8adab73a36e9afc12cb08c99c551ce02e97315de2a288d7",
        3145856,
        "f17ee07bb9b513770b0a98f53c53440957de3b04b162a23991cc92a64cc678ce",
        3145856,
        "a302df41411a3d947b1a3378f6eb64df763726e84470acd9c47fcaa02f7f4d21",
    ),
    (
        "view-006",
        "indoors",
        "scene_00020",
        "scan_00186",
        "00020_00186_indoors_000_000",
        1194403,
        "2478882bbac47733a075b3871d178e4c27ba432d0f6d2af1341265a8eecf1369",
        3145856,
        "b4c7533123e6c7204e6b67d4494af0e59bb454b476e552539c2b058509fc2f06",
        3145856,
        "dbf9d3e124e71fe633411f6e54d58f712095044dd152e3f8d18cbd5c75472c42",
    ),
    (
        "view-007",
        "indoors",
        "scene_00020",
        "scan_00186",
        "00020_00186_indoors_020_000",
        1171589,
        "5a7ffa68320043e62c35319b8b2e971bd29331efef181e92ca04cac17a9f4605",
        3145856,
        "7b488f809c23c62594406d1e8fafb4857d7da7d6c98571cc0e3a4a10b76bbd71",
        3145856,
        "403d0dbaa52767792e69a4471c03a6c0dca174dd517dc8c567dd05851c4b3528",
    ),
    (
        "view-008",
        "indoors",
        "scene_00020",
        "scan_00187",
        "00020_00187_indoors_000_000",
        1258618,
        "16bf17dce5d4025536bada42024cd756c7e6f6ee5012a7e7c606039c7e7028db",
        3145856,
        "77068b8f5e9faa2e6cc89f381720873a36b5631643f3df4cff40d8f59d10ed28",
        3145856,
        "28d3e14c26011438587539d9e5c4e9fecbd631f02727fd35ac5f533cbabfb8fb",
    ),
    (
        "view-009",
        "indoors",
        "scene_00020",
        "scan_00187",
        "00020_00187_indoors_000_020",
        1202574,
        "e241249dadc41376befa417e33165412a834841334cee9e950729eaecb1243ce",
        3145856,
        "3ee01eb76928b1cdc49e81a9a7af29797fb4d25c5409754314972da29ed9f9dd",
        3145856,
        "1ec90f9292cc265e1bd8865757fc09cd119ba4ec9c6cb2863803823b46cc994f",
    ),
    (
        "view-010",
        "indoors",
        "scene_00021",
        "scan_00188",
        "00021_00188_indoors_090_000",
        981100,
        "1040b93aa913ad042231e9b9809293745b02c24c90c0cba706a44c77b7847f56",
        3145856,
        "b4a9073b489bff9d472576592185733ee7a709a7aa04445f0f43356ee333e248",
        3145856,
        "940df9212e7cde2ba9225848f57f3aa8d15d6c9dfe2f63bbf3aae00e5c09c282",
    ),
    (
        "view-011",
        "indoors",
        "scene_00021",
        "scan_00188",
        "00021_00188_indoors_090_020",
        1057542,
        "ffe1f5390dca8b043d672a568d28b4f3409276dce13da76f3d18b16e56bb85a9",
        3145856,
        "d7a5487729ca00dc8ce1f0c08006246143e34b79e8da211b0b4ce81a21db23aa",
        3145856,
        "2922ae237dc3d890ed80d0cd8d704c61728ced3383cf249c24d2c36246212853",
    ),
    (
        "view-012",
        "indoors",
        "scene_00021",
        "scan_00189",
        "00021_00189_indoors_000_000",
        1036343,
        "fed8f8be2ea611b2d1ce2b75440f029d6b786cef7218bace273e96dc79acf60b",
        3145856,
        "e71a79b0a7a22fb6a979543809abb2bd7f906487a7b2e9078d4d93cfc562eed7",
        3145856,
        "4b053fdf65b2123bdcadb43e3629521a8eacce52d7fcd820ccddfb8f3ae39663",
    ),
    (
        "view-013",
        "indoors",
        "scene_00021",
        "scan_00189",
        "00021_00189_indoors_000_020",
        1075564,
        "8dd7d9980c7c79ea74d5438154cbe83fc6e245189b9d36406fc1509d8b7b32e9",
        3145856,
        "9b5f1be74b6fc3901fffe7dc38be3d3a2662ad92a565e9b1e263d2b5784bbb94",
        3145856,
        "625221f96afa15cf03e409dcd04910c793f2a187d85b822f834fead3ae601da9",
    ),
    (
        "view-014",
        "indoors",
        "scene_00021",
        "scan_00190",
        "00021_00190_indoors_050_000",
        1000911,
        "4d90cc37cf28095d1aa45668f6f6457eca7f28560a0f62b640b0802a1f5394eb",
        3145856,
        "c939b634fb09d8a08a111bb6739e4298245dea12171becff202d900b72284fc7",
        3145856,
        "39e2709c2a01a290103bf55f48d4674ff08e213ac7f42afe7cc0e62cc43bad68",
    ),
    (
        "view-015",
        "indoors",
        "scene_00021",
        "scan_00190",
        "00021_00190_indoors_050_030",
        996884,
        "851a7f44cd3ebfca766a3bd25a818a000277f2752f16c0e1bac0655ba77f6168",
        3145856,
        "1d11b9096ab6eeca053307d39eb2a7c3db9d1cc529b5c243421c111eca6ea0a1",
        6291584,
        "8192c4b01dd2c6776cc03f68e9a61f792c9149e27ea7959e362c224193a85349",
    ),
    (
        "view-016",
        "indoors",
        "scene_00021",
        "scan_00191",
        "00021_00191_indoors_000_040",
        1057445,
        "2a7fafdbdfbaf1bbb9363b32b5337a86f391ebdd8cb23a0227d5e6ec3bddc11d",
        3145856,
        "64609dc44db5769caae0aa5b5a29a92a380ff5aaf9c0173c172a7dda0cd8f42a",
        3145856,
        "7c2ee20e3128526b51067419562fba26d25ba734d2a92a31467e9b42d190f0a8",
    ),
    (
        "view-017",
        "indoors",
        "scene_00021",
        "scan_00191",
        "00021_00191_indoors_010_030",
        1053502,
        "3a686ca893ba52f2780370737b762080b2cf2ce7201c4311a9f31ed29595f92f",
        3145856,
        "ece8e9acdeb8601bc111d70b6260d0ea92cea926166dc5b0c1d2bde3f5406b8e",
        3145856,
        "bdf32493a0ad439062821d9f9f3f3a966454c431d2c4bd4b5d22be89bceb494a",
    ),
    (
        "view-018",
        "indoors",
        "scene_00021",
        "scan_00192",
        "00021_00192_indoors_000_000",
        1061846,
        "28c6d3acbf088975eddc38ab6e5d4162b7f93fba8f5459d0f833c2d30fe89530",
        3145856,
        "ebbec603a9625ee58cfd861770402c1cd9fafeb5b6126e6a9a7628941bddefce",
        3145856,
        "0eeeadfb8976b9a77f3ce4351401bec8b1e61ba0a4b2a8397e7328dae738cde5",
    ),
    (
        "view-019",
        "indoors",
        "scene_00021",
        "scan_00192",
        "00021_00192_indoors_000_020",
        1057090,
        "fe69baf50e4801abd3b5f9b2dacd70e33a0ed8c8f4a2d975c980128bf1802e74",
        3145856,
        "2aba7a7f71b5dbd799a42609aab58ae2b6ad7ca7521dc05db5288b05010b585c",
        3145856,
        "1263a8da470fcdaad3be5d70034adeac2f2d7b8aa862fa1f962ab908e5a2184f",
    ),
    (
        "view-020",
        "outdoor",
        "scene_00022",
        "scan_00193",
        "00022_00193_outdoor_000_000",
        1260962,
        "f513f607fbdcdbe3545cfc67d90828d81619be157689c1db26826559c1cae0a6",
        3145856,
        "a145d0f58dfabc71e59bb5831d42b11989d2be3d8bb6ef16ca16449f77907b41",
        3145856,
        "c06249a9c2d6b204fba67219db98be321f5fa39707b67d3b2e3880fe6a707c55",
    ),
    (
        "view-021",
        "outdoor",
        "scene_00022",
        "scan_00193",
        "00022_00193_outdoor_000_020",
        1387923,
        "2fa8096f650f9bb7327cffa8079d471feaffd11b1a37b3133c456d4bb89356de",
        3145856,
        "59513f5e83be4ab75dd8a996704fcbb32f1a4b1143c874654d6106781b7386b7",
        3145856,
        "1ec6d6f79ba218282caf7ae1220b0fa19d06e56e09458b0a85e0825a91545049",
    ),
    (
        "view-022",
        "outdoor",
        "scene_00022",
        "scan_00194",
        "00022_00194_outdoor_000_000",
        1386881,
        "01786324ca6c763a67e32adec512d83d32c295e3243d23fce9fa1b56f8031eb3",
        3145856,
        "84024306218657d9eea51c2abd54044e6ee6293d0193a7e185ed98b1f302b91e",
        6291584,
        "7064a6740d486b03a771a14d9801807dfd6dc2e48ac2c0f42417e755a63f067a",
    ),
    (
        "view-023",
        "outdoor",
        "scene_00022",
        "scan_00194",
        "00022_00194_outdoor_000_020",
        1550903,
        "3e759d88c101e158a81eecf16b87c6e96b4ecd3d5d6e912a46af789a1cfd2f1a",
        3145856,
        "1aa6fd2ef08489b8d0a8547a2fc6093936f524df56967d797363baad7c6f0fff",
        3145856,
        "933093e8d49a222a2329f89d4adb82a8e4eb9e8a72079d885e8eebf97f31f801",
    ),
    (
        "view-024",
        "outdoor",
        "scene_00022",
        "scan_00195",
        "00022_00195_outdoor_000_000",
        1354956,
        "485ae591b4e9197422d23834e658d3b7c0b2a9a8a1f4d190dc4e8285abcf4b58",
        3145856,
        "a0eb3b67dbdb6c7a541bc787f8b2c83c927b510477b83ea8eb47e0c0b64520fc",
        3145856,
        "1425f824f4bd3bfd4bc39f1ef6cff65c24578a79a6a5e56b1813fd68e6405daa",
    ),
    (
        "view-025",
        "outdoor",
        "scene_00022",
        "scan_00195",
        "00022_00195_outdoor_000_020",
        1312039,
        "b505cf1a9e05be9caecc3141b3f6628e2d52aef7bd6768e0495d1fe68f022cc7",
        3145856,
        "00ca098eb81e758870344c0ea0671c15a75964ebc9afa82e5c842e9edf465b1b",
        3145856,
        "9c3b3f29d5c27d8ab1a899528fd98d48a608422f5d6878440c56e9452209082d",
    ),
    (
        "view-026",
        "outdoor",
        "scene_00022",
        "scan_00196",
        "00022_00196_outdoor_000_010",
        1548233,
        "aa7cc5973a301d69420c74169d70d850d206adab453a1677c81ad3df8e456c01",
        3145856,
        "d415a6bb86593c6bb35a6457bbc13cd451206907e6225ca7d37f0171a5a19dc9",
        3145856,
        "63c640cb2c1d438832cacb039a307141e1a5e426d2f3fd2961e2b4e566215e47",
    ),
    (
        "view-027",
        "outdoor",
        "scene_00022",
        "scan_00196",
        "00022_00196_outdoor_000_030",
        1581362,
        "64683db027716e6d5564c8962e2ef819be8e992a7514c5f0f3f42fa056230504",
        3145856,
        "bee31f7ca3020b9415e3debff8c219ee3502b104b1844d438a51d84a199979e9",
        3145856,
        "1aec43f5993a614a4f2b6d7577739615fc7780f13a49fbf899ee24753132e5a0",
    ),
    (
        "view-028",
        "outdoor",
        "scene_00022",
        "scan_00197",
        "00022_00197_outdoor_000_000",
        1397626,
        "7b491f25a7a26e435ef5feb4d90ce5929954a2473e34f76bd70cc2d9a81251c3",
        3145856,
        "5045c2038970c4c282ffccb141c3afef9027a71523d4796189c59a544abc8847",
        3145856,
        "a80a8aa9ecadbef495966bc08935e8e02035c382c8ff8d75075646ee14e00191",
    ),
    (
        "view-029",
        "outdoor",
        "scene_00022",
        "scan_00197",
        "00022_00197_outdoor_010_010",
        1427912,
        "ab669c4bf03ca844c125cb3e6d963364f5f14b36d0af9288130d85a174340205",
        3145856,
        "cd22ceb5a19427d36e1199e167237b0626b468947aa6ec21ef2889b3f687a42e",
        3145856,
        "69912b6858ed773c453b90993ef0c886b69c3bb3ee43644e54682b1e4f11f4a5",
    ),
    (
        "view-030",
        "outdoor",
        "scene_00023",
        "scan_00198",
        "00023_00198_outdoor_000_020",
        1196280,
        "8e058762aeb6d508cc4a453bb51bb4195596644fdce856199e8ff47289154163",
        3145856,
        "07f0d1b82b79e37df0afa504b15061a1d801ffb7143da44d98c7e4573c126b30",
        3145856,
        "fdcca8151892640e098085128f1e34786f79c9ebd8f01c7f14f67aeff5ee529a",
    ),
    (
        "view-031",
        "outdoor",
        "scene_00023",
        "scan_00198",
        "00023_00198_outdoor_070_020",
        1153475,
        "2c6ba4a04b24415855c6d56f9e424e752682c9598ed0a6961a2c095c0fde80e7",
        3145856,
        "0e1cef73cd8aa9611eee6a36395bda320bdbc5608d9431bcce24e59a1e0908d1",
        6291584,
        "4571c9f2b44300fbc1a4152b669720d367860f01827cc5512fd939f24965b7a6",
    ),
    (
        "view-032",
        "outdoor",
        "scene_00023",
        "scan_00199",
        "00023_00199_outdoor_000_020",
        1443825,
        "065fdaf432f60cab7d4d85f77f6e8a8da2d8d7f6b618173e61b71c4db79b1b14",
        3145856,
        "734166fea6643e708cda8075833cffb604a288f4b8790e2c5d758deacb7867d9",
        3145856,
        "ee02105445049bd21b66b642dae6da7992d410b4a5f7c17397dfef192e12b246",
    ),
    (
        "view-033",
        "outdoor",
        "scene_00023",
        "scan_00199",
        "00023_00199_outdoor_010_000",
        1401671,
        "27ac9a575d259ce2511a02b9c5265e39dcbebe4afd39e9f8b64a296ef6ea1095",
        3145856,
        "303990f57fcd277a43df832f1f03ef42a78008f99d0af024fe3e4ed53c977db4",
        3145856,
        "94c855bbefce29e551ceeb0b9971fcd6fa20adee845ab212147d694f6362d813",
    ),
    (
        "view-034",
        "outdoor",
        "scene_00023",
        "scan_00200",
        "00023_00200_outdoor_000_010",
        1103637,
        "3d1c0f779183fe7eb383e05237a5462dc18fd790f35a80e90787e94d0e43c1c7",
        3145856,
        "b515576843d3ad6bd718d8e578c7e3bd128f82bcdadc51d2855ce77f65c63279",
        3145856,
        "23c79f29ae3ea4c3cc4f4e4cbfc4316b5e86a8fb661d1b2cfb2ddaf88f3e12de",
    ),
    (
        "view-035",
        "outdoor",
        "scene_00023",
        "scan_00200",
        "00023_00200_outdoor_000_050",
        1225980,
        "6046d35e5454ceac3691c5086d262431260b9839194be58fd9015415f21b5dba",
        3145856,
        "672aa46a95e17433425bd0510f502eebb637fe494db3afe41c36b818ab3d7334",
        3145856,
        "1434e085120fca95356d956046c0b55120d0b4a32a5ec14394d67104f38eb1eb",
    ),
    (
        "view-036",
        "outdoor",
        "scene_00024",
        "scan_00201",
        "00024_00201_outdoor_000_000",
        1916342,
        "2fd43a35faee39cdcf8b08b1f23915c8aabae57601e1c4eb90ac98c2726f6726",
        3145856,
        "7585ead8e9e351d703220c1fb4aa1a070874e0afcdb676646e8db7e90c57d46a",
        3145856,
        "4e6f8a5647539f94a7980255bb7c0be7dfdb7fe30395c3fccfdd47fcbddcb734",
    ),
    (
        "view-037",
        "outdoor",
        "scene_00024",
        "scan_00201",
        "00024_00201_outdoor_000_020",
        1842876,
        "08c4b8342eea57a08d4380108ea14f342b43af3ef10fd7d1f9ee28dac1a2af6b",
        3145856,
        "c15fbde3764e85cb2c2b88a02562101447141b8ed63f0a70fa6da0c8a22a69a1",
        3145856,
        "5a92fbe940ba37d595197c9caa012526ec6b2f32ebceb5ffadb3b98ecafb80c3",
    ),
    (
        "view-038",
        "outdoor",
        "scene_00024",
        "scan_00202",
        "00024_00202_outdoor_000_030",
        1911126,
        "29640b6d05b7f6f310c31a9a26e160518f679439bf619983dbe0b5fd1afb39aa",
        3145856,
        "a85149326d869f091b5611e7af0dfe0b0dad770bd8eb92f21bf4be2c76a65887",
        3145856,
        "f138b3fc58846961bb7193825be86a427d7689cf1f04b23a9eb830988eb93d84",
    ),
    (
        "view-039",
        "outdoor",
        "scene_00024",
        "scan_00202",
        "00024_00202_outdoor_070_010",
        1877641,
        "e6c78b21fdd07ae5feeb21beb3c5f391522f041c9ddb55ba53a47cfd5994e434",
        3145856,
        "942e5fb2d5e63738fbc09c978d6f8ac878f50313c99bbd73ffc9239d0f44f109",
        3145856,
        "37ddc9e81d1c9f6abee51157213f4a8d5579ad4994bf5982220a8dc023dcd05a",
    ),
)

DEFAULT_CACHE_DIR = Path("weights") / "diode-sample"  # working-directory-relative, like the notebook
SAMPLE_SEED = 42
SAMPLE_SPLIT = {"train": 6, "validation": 2, "test": 2}  # scans per domain; 2 domains x 2 views -> 24 / 8 / 8
MIN_RECORDS = 4
MAX_RECORDS = 2_000
MIN_VALID_FRACTION = 0.05  # a depth map must label at least this fraction of its pixels
MAX_DEPTH_M = 1_000.0
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def file_url(domain: str, scene: str, scan: str, stem: str, suffix: str) -> str:
    return f"{CORPUS_BASE_URL}{domain}/{scene}/{scan}/{stem}{suffix}"


def _pins(record: tuple) -> dict[str, tuple[int, str]]:
    return {
        ".png": (record[5], record[6]),
        "_depth.npy": (record[7], record[8]),
        "_depth_mask.npy": (record[9], record[10]),
    }


def fetch_corpus(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> dict[str, dict[str, bytes]]:
    """Return every pinned file (bytes keyed by record id, then suffix) from the cache or the Hub mirror."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    out: dict[str, dict[str, bytes]] = {}
    for record in SAMPLE_RECORDS:
        rid, domain, scene, scan, stem = record[:5]
        out[rid] = {}
        for suffix, (size, digest) in _pins(record).items():
            local = cache / f"{stem}{suffix}"
            data = local.read_bytes() if local.is_file() else b""
            if len(data) != size or _sha256_bytes(data) != digest:
                url = file_url(domain, scene, scan, stem, suffix)
                if fetcher is not None:
                    data = fetcher(url)
                else:
                    request = urllib.request.Request(
                        url, headers={"User-Agent": "dimer-depth-anything-tutorial/1.0"}
                    )
                    with urllib.request.urlopen(request, timeout=300) as response:  # noqa: S310 (pinned https URL)
                        data = response.read()
                if len(data) != size or _sha256_bytes(data) != digest:
                    raise ValueError(
                        f"{rid} ({stem}{suffix}): fetched {len(data)} bytes with sha256 "
                        f"{_sha256_bytes(data)[:16]}…, pinned {size} / {digest[:16]}…"
                    )
                local.write_bytes(data)
            out[rid][suffix] = data
    return out


def _load_npy(data: bytes) -> np.ndarray:
    return np.load(io.BytesIO(data), allow_pickle=False)


def read_corpus(files: Mapping[str, Mapping[str, bytes]]) -> list[dict[str, Any]]:
    """Decode the verified files into `{id, image, depth, mask}` records with their provenance."""
    out = []
    for record in SAMPLE_RECORDS:
        rid, domain, scene, scan, stem = record[:5]
        if rid not in files:
            raise ValueError(f"corpus is missing {rid}")
        image = Image.open(io.BytesIO(files[rid][".png"]))
        image.load()
        depth = _load_npy(files[rid]["_depth.npy"])
        depth = depth[..., 0] if depth.ndim == 3 else depth
        mask = _load_npy(files[rid]["_depth_mask.npy"]) > 0
        out.append(
            {
                "id": rid,
                "image": image.convert("RGB"),
                "depth": np.ascontiguousarray(depth, dtype=np.float32),
                "mask": np.ascontiguousarray(mask, dtype=bool),
                "domain": domain,
                "scene": scene,
                "scan": scan,
                "source_stem": stem,
                "source_url": file_url(domain, scene, scan, stem, ".png"),
            }
        )
    return out


def build_sample_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded draw of whole scans per domain: `sizes` = scans per domain for train / validation / test."""
    sizes = dict(sizes or SAMPLE_SPLIT)
    rng = random.Random(seed)
    by_scan: dict[tuple[str, str], list[dict[str, Any]]] = {}
    for record in records:
        by_scan.setdefault((str(record.get("domain", "")), str(record["scan"])), []).append(dict(record))
    out: dict[str, list[dict[str, Any]]] = {name: [] for name in sizes}
    for domain in sorted({d for d, _ in by_scan}):
        scans = sorted(s for d, s in by_scan if d == domain)
        rng.shuffle(scans)
        needed = sum(sizes.values())
        if len(scans) < needed:
            raise ValueError(f"{domain}: only {len(scans)} scans available, need {needed}")
        cursor = 0
        for name, per_domain in sizes.items():
            for scan in scans[cursor : cursor + per_domain]:
                out[name].extend(by_scan[(domain, scan)])
            cursor += per_domain
    for name in out:
        rng.shuffle(out[name])
        out[name] = [{**r, "id": f"{name}-{i:03d}", "source_id": r["id"]} for i, r in enumerate(out[name])]
    return out


def fetch_sample_dataset(
    *,
    cache_dir: str | Path | None = None,
    fetcher: Any = None,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """The tutorial splits from the pinned corpus."""
    return build_sample_dataset(
        read_corpus(fetch_corpus(cache_dir=cache_dir, fetcher=fetcher)), seed=seed, sizes=sizes
    )


def _as_array(value: Any, label_name: str, key: str) -> np.ndarray:
    if isinstance(value, str | Path):
        path = Path(value)
        if not path.is_file():
            raise ValueError(f"{label_name}: {key} file not found: {path}")
        value = np.load(path, allow_pickle=False)
    if not isinstance(value, np.ndarray):
        raise ValueError(f"{label_name}: {key} must be a numpy array or a path to a .npy file")
    if value.ndim == 3 and value.shape[-1] == 1:
        value = value[..., 0]
    if value.ndim != 2:
        raise ValueError(f"{label_name}: {key} must be a 2-D H x W array, got shape {value.shape}")
    return value


def _check_record(record: Any, index: int) -> dict[str, Any]:
    label_name = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{label_name} must be a mapping with id/image/depth[/mask]")
    for key in ("id", "image", "depth"):
        if key not in record:
            raise ValueError(f"{label_name} is missing {key!r}")
    rid, image = record["id"], record["image"]
    if not isinstance(rid, str) or not _ID_RE.match(rid):
        raise ValueError(f"{label_name}: id must match {_ID_RE.pattern}")
    if isinstance(image, str | Path):
        path = Path(image)
        if not path.is_file():
            raise ValueError(f"{label_name}: image file not found: {path}")
        image = Image.open(path)
        image.load()
    if not isinstance(image, Image.Image):
        raise ValueError(f"{label_name}: image must be a PIL.Image.Image or a file path")
    width, height = image.size
    short, long = min(width, height), max(width, height)
    if short < MIN_IMAGE_SIDE or long > MAX_IMAGE_SIDE:
        raise ValueError(
            f"{label_name}: image side outside {MIN_IMAGE_SIDE}..MAX_IMAGE_SIDE={MAX_IMAGE_SIDE} px: "
            f"{image.size}"
        )
    if long / short > MAX_ASPECT_RATIO:
        raise ValueError(
            f"{label_name}: aspect ratio {long / short:.2f} > MAX_ASPECT_RATIO {MAX_ASPECT_RATIO}"
        )
    depth = _as_array(record["depth"], label_name, "depth")
    if depth.shape != (height, width):
        raise ValueError(
            f"{label_name}: depth shape {depth.shape} != image (height, width) {(height, width)}"
        )
    if not np.issubdtype(depth.dtype, np.number):
        raise ValueError(f"{label_name}: depth must be numeric, got {depth.dtype}")
    depth = np.ascontiguousarray(depth, dtype=np.float32)
    if record.get("mask") is None:
        mask = np.isfinite(depth) & (depth > 0)
    else:
        mask = _as_array(record["mask"], label_name, "mask")
        if mask.shape != depth.shape:
            raise ValueError(f"{label_name}: mask shape {mask.shape} != depth shape {depth.shape}")
        mask = np.ascontiguousarray(mask, dtype=bool) & np.isfinite(depth) & (depth > 0)
    fraction = float(mask.mean())
    if fraction < MIN_VALID_FRACTION:
        raise ValueError(
            f"{label_name}: only {fraction:.1%} of pixels carry valid depth; "
            f"at least {MIN_VALID_FRACTION:.0%} are required"
        )
    if float(depth[mask].max()) > MAX_DEPTH_M:
        raise ValueError(f"{label_name}: depth exceeds MAX_DEPTH_M={MAX_DEPTH_M} m (metres are expected)")
    item = {"id": rid, "image": image.convert("RGB"), "depth": depth, "mask": mask}
    for key in ("source_id", "domain", "scene", "scan", "group", "source_stem", "source_url"):
        if key in record:
            item[key] = record[key]
    return item


def validate_dataset(
    records: Sequence[Mapping[str, Any]], *, min_records: int = MIN_RECORDS, max_records: int = MAX_RECORDS
) -> dict[str, Any]:
    """Structural validation of a depth-labelled image dataset; raises ValueError before any model import."""
    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, str | bytes):
        raise ValueError("records must be a list of {id, image, depth[, mask]} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    checked = [_check_record(record, index) for index, record in enumerate(records)]
    ids = [r["id"] for r in checked]
    if len(set(ids)) != len(ids):
        duplicate = next(i for i in ids if ids.count(i) > 1)
        raise ValueError(f"duplicate id {duplicate!r}")
    sides = [max(r["image"].size) for r in checked]
    valid = [float(r["mask"].mean()) for r in checked]
    depths = [float(np.median(r["depth"][r["mask"]])) for r in checked]
    domains: dict[str, int] = {}
    for r in checked:
        domains[str(r.get("domain", "unspecified"))] = domains.get(str(r.get("domain", "unspecified")), 0) + 1
    return {
        "records": checked,
        "n_records": len(checked),
        "domain_counts": domains,
        "scans": len({str(r.get("scan", r.get("group", r["id"]))) for r in checked}),
        "image_side": {"min": min(sides), "max": max(sides)},
        "valid_fraction": {"min": round(min(valid), 4), "max": round(max(valid), 4)},
        "median_depth_m": {"min": round(min(depths), 3), "max": round(max(depths), 3)},
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def image_digest(image: Image.Image) -> str:
    """SHA-256 of the decoded RGB pixels (size + bytes), so a re-encoded copy of the same photo matches."""
    rgb = image.convert("RGB")
    return _sha256_bytes(f"{rgb.size[0]}x{rgb.size[1]}:".encode() + rgb.tobytes())


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    payload = [
        [
            r["id"],
            image_digest(r["image"]),
            _sha256_bytes(np.ascontiguousarray(r["depth"], dtype=np.float32).tobytes()),
        ]
        for r in records
    ]
    return _sha256_bytes(json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8"))


def _split_unit(record: Mapping[str, Any]) -> str:
    return str(record.get("scan") or record.get("group") or record.get("source_id") or record["id"])


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no image (by decoded-pixel digest) and no scan appears in two splits (leakage check)."""
    seen: dict[str, str] = {}
    scans: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = image_digest(record["image"])
            if key in seen and seen[key] != name:
                raise ValueError(f"image {record['id']!r} appears in both {seen[key]} and {name}")
            seen[key] = name
            unit = _split_unit(record)
            if unit in scans and scans[unit] != name:
                raise ValueError(f"scan {unit!r} has views in both {scans[unit]} and {name}")
            scans[unit] = name
    return {name: len(records) for name, records in splits.items()}


def scan_summary(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Scans and domains per split (an observation of what the split unit was)."""
    out: dict[str, Any] = {}
    for name, records in splits.items():
        out[name] = {
            "views": len(records),
            "scans": len({_split_unit(r) for r in records}),
            "domains": {
                d: sum(1 for r in records if str(r.get("domain", "unspecified")) == d)
                for d in sorted({str(r.get("domain", "unspecified")) for r in records})
            },
        }
    return out


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.15,
    test_fraction: float = 0.2,
    seed: int = 0,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded shuffle of a BYOD dataset into train/validation/test by split unit (`scan` / `group`, else the
    image itself) after de-duplicating images, so views of one scan never straddle splits."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records)["records"]
    seen: set[str] = set()
    by_unit: dict[str, list[dict[str, Any]]] = {}
    for record in checked:
        key = image_digest(record["image"])
        if key not in seen:
            seen.add(key)
            by_unit.setdefault(_split_unit(record), []).append(record)
    rng = random.Random(seed)
    units = sorted(by_unit)
    rng.shuffle(units)
    n_test = max(1, round(len(units) * test_fraction))
    n_val = round(len(units) * val_fraction)
    splits: dict[str, list[dict[str, Any]]] = {"test": [], "validation": [], "train": []}
    for name, chosen in (
        ("test", units[:n_test]),
        ("validation", units[n_test : n_test + n_val]),
        ("train", units[n_test + n_val :]),
    ):
        for unit in chosen:
            splits[name].extend(by_unit[unit])
    for part in splits.values():
        rng.shuffle(part)
    if len(splits["train"]) < MIN_RECORDS:
        raise ValueError(
            f"split leaves {len(splits['train'])} training records; at least {MIN_RECORDS} are required"
        )
    return splits


def load_byod_dataset(path: str | Path, *, require_group: bool = True) -> list[dict[str, Any]]:
    """Read `{id, image, depth[, mask]}` records from a directory or a zip holding `records.csv` (columns
    `id`, `image`, `depth`, `group`, optional `mask`) beside the files; depth and mask are `.npy` arrays
    (metres; boolean) decoded from the archive, never extracted to disk. `group` (the scan, capture session
    or device the view belongs to) must be non-empty on every row unless `require_group=False`, in which
    case the split falls back to one unit per view and the group-disjoint guarantee is gone."""
    source = Path(path)
    if source.is_dir():
        table = (source / "records.csv").read_text(encoding="utf-8")
        loader = lambda name: (source / name).read_bytes()  # noqa: E731
    elif source.is_file() and source.suffix.lower() == ".zip":
        archive = zipfile.ZipFile(source)
        members = {Path(n).name: n for n in archive.namelist()}
        if "records.csv" not in members:
            raise ValueError("BYOD zip must contain records.csv")
        table = archive.read(members["records.csv"]).decode("utf-8")
        loader = lambda name: archive.read(members[name])  # noqa: E731
    else:
        raise ValueError("BYOD datasets must be a directory or a .zip holding records.csv and the files")
    rows = list(csv.DictReader(io.StringIO(table)))
    missing = {"id", "image", "depth"} - set(rows[0].keys() if rows else set())
    if missing:
        raise ValueError(f"records.csv is missing columns {sorted(missing)}")
    out = []
    seen: set[str] = set()
    for row in rows:
        group = (row.get("group") or "").strip()
        if require_group and not group:
            raise ValueError(
                f"records.csv row for id {row['id']!r} has no `group`; every row needs the scan, capture "
                "session or device the view belongs to so the split stays group-disjoint (pass "
                "require_group=False to split by view instead, without that guarantee)"
            )
        if row["id"] in seen:
            raise ValueError(f"records.csv lists id {row['id']!r} more than once")
        seen.add(row["id"])
        image = Image.open(io.BytesIO(loader(row["image"])))
        image.load()
        item: dict[str, Any] = {
            "id": row["id"],
            "image": image.convert("RGB"),
            "depth": _load_npy(loader(row["depth"])),
        }
        if row.get("mask"):
            item["mask"] = _load_npy(loader(row["mask"]))
        if group:
            item["group"] = group
        out.append(item)
    return out


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """Write the records table of a split (id, image, depth, mask, group, provenance) as BYOD expects it."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(
            handle, fieldnames=["id", "image", "depth", "mask", "group", "domain", "source_url"]
        )
        writer.writeheader()
        for record in records:
            stem = record.get("source_stem") or record["id"]
            writer.writerow(
                {
                    "id": record["id"],
                    "image": f"{stem}.png",
                    "depth": f"{stem}_depth.npy",
                    "mask": f"{stem}_depth_mask.npy",
                    "group": _split_unit(record),
                    "domain": record.get("domain", ""),
                    "source_url": record.get("source_url", ""),
                }
            )
    return out

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `4`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `5426e4f0f365…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `DepthAnythingPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "depth-anything-v2-small",
  "modelId": "depth-anything/Depth-Anything-V2-Small-hf",
  "revision": "5426e4f0f36572d16453bbda7a8389317b1bef99",
  "files": [
    {
      "path": "README.md",
      "bytes": 4400,
      "sha256": "319566441daaac0c5ed83e75a8fd19bae0863f3275e23f147c06fe0906edf6fb"
    },
    {
      "path": "config.json",
      "bytes": 950,
      "sha256": "c56698d3643dde1f83ea2212759e6b31a22b8f827246a36dd007ee8a22b3ff75"
    },
    {
      "path": "model.safetensors",
      "bytes": 99173660,
      "sha256": "3152477ce0d8d6978d76b995120de97cb5b928701fd0f817769f59e249a16b70"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 775,
      "sha256": "d41175c0d889477ca8fc67191e540faef14baf6275157b3fdecf78469e6bbf84"
    }
  ],
  "totalBytes": 99179785
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = DepthAnythingPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Referenced corpus, validation and scan-level split

`fetch_corpus` downloads the 120 pinned files (or reads them from the cache), refuses a byte-size or SHA-256 mismatch per file before it is decoded, and `read_corpus` turns each view into a `{id, image, depth, mask}` record with its domain, scene, scan and source URL. `build_sample_dataset` draws whole scans per domain — 6 training, 2 validation and 2 test scans from each of the indoor and outdoor sets — by a seeded shuffle, so the 24 / 8 / 8 views never share a scene across splits; `validate_dataset` then checks every record against the contract, `check_split_disjoint` asserts no view (by decoded-pixel digest) and no scan appears in two splits, `scan_summary` reports scans and domains per split, and the training records table is written to `outputs/depth_anything_depth_estimation_train.csv` in the shape BYOD expects.

Look for: 40 views, 20 scans of 2, splits 24 / 8 / 8 with 12 / 4 / 4 scans, three digests, median depths from about a metre indoors to tens of metres outdoors, and four refusal probes — a duplicate id, a depth map of the wrong shape, a mask with no valid pixels and a dataset too small to train on — each rejected before `torch` does anything. About a minute on the first run for the 312 MB download.

In [ ]:
import hashlib
import io
import json
import time

USE_BYOD = False  # @param {type:"boolean"}
SPLIT_SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
t0 = time.perf_counter()
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_path = Path('work') / file_name
    byod_path.parent.mkdir(parents=True, exist_ok=True)
    byod_path.write_bytes(payload)
    records = load_byod_dataset(byod_path)
    splits = split_dataset(records, seed=SPLIT_SEED)
    data_source = 'BYOD (' + file_name + ')'
    raw_count = {'byod': len(records)}
else:
    corpus = read_corpus(fetch_corpus(cache_dir='weights/diode-sample'))
    raw_count = {'views': len(corpus), 'scans': len({(r['domain'], r['scan']) for r in corpus}), 'domains': sorted({r['domain'] for r in corpus})}
    splits = build_sample_dataset(corpus, seed=SPLIT_SEED)
    data_source = f'{CORPUS_NAME} ({CORPUS_RELEASE}; {CORPUS_LICENSE})'
fetch_seconds = round(time.perf_counter() - t0, 1)
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
dataset_manifests = {name: validate_dataset(part, min_records=1) for name, part in splits.items()}
disjoint = check_split_disjoint(splits)
summary = scan_summary(splits)
write_dataset_csv(train_records, 'outputs/depth_anything_depth_estimation_train.csv')
print({'data_source': data_source, 'raw': raw_count, 'splits': disjoint, 'scan_summary': summary, 'fetch_seconds': fetch_seconds, 'corpus_bytes': CORPUS_BYTES})
for name, manifest in dataset_manifests.items():
    print({name: {'n': manifest['n_records'], 'scans': manifest['scans'], 'domains': manifest['domain_counts'], 'image_side': manifest['image_side'], 'valid_fraction': manifest['valid_fraction'], 'median_depth_m': manifest['median_depth_m'], 'digest': manifest['digest'][:16] + '...'}})
example = train_records[0]
print({'example': {k: example[k] for k in ('id', 'domain', 'scene', 'scan', 'source_url') if k in example}, 'size': example['image'].size, 'depth_dtype': str(example['depth'].dtype), 'valid_fraction': round(float(example['mask'].mean()), 4)})

probes = {
    'duplicate id': [{**r, 'id': 'same'} for r in train_records[:4]],
    'depth shape': [{**train_records[0], 'depth': train_records[0]['depth'][:-1]}, *train_records[1:4]],
    'no valid depth': [{**train_records[0], 'mask': np.zeros_like(train_records[0]['mask'])}, *train_records[1:4]],
    'too small': train_records[:3],
}
for name, probe in probes.items():
    try:
        validate_dataset(probe)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. Predict through the inference contract, with a metric reference

Before any adaptation, the inference contract is exercised as it always was, on one test view. `validate_inputs` applies exactly the checks `predict` applies — type, sides 14..`MAX_IMAGE_SIDE` px, aspect ratio at most `MAX_ASPECT_RATIO` — and returns an input manifest; a deliberately over-wide image is validated too and its rejection recorded as a finding. `predict` returns relative inverse depth at the input resolution with four sanity checks. Because this view carries metric depth from a laser scanner, `evaluation_report` can for the first time return **`sample-sanity`**: it aligns the prediction to the reference by least squares in inverse depth (the pipeline's own `abs_rel`, invalid pixels zeroed out of the reference) and reports AbsRel for this one image — sanity evidence, not a benchmark. The build record's probe view — an outdoor scan — scored 0.12; indoor views scored 0.04–0.13 and outdoor views with sky and distant structure up to 0.40. A side-by-side preview (image | predicted inverse depth) is displayed.

In [ ]:
probe_record = test_records[0]
image = probe_record['image']
print({'ceilings': {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_ASPECT_RATIO': MAX_ASPECT_RATIO}, 'contract': {'DEPTH_KIND': DEPTH_KIND, 'TRANSFORMER_BLOCKS': TRANSFORMER_BLOCKS, 'PARAMETER_COUNT': PARAMETER_COUNT, 'EVAL_DEPTH_RANGE_M': EVAL_DEPTH_RANGE_M}})
input_manifest = validate_inputs(image, names=[probe_record['id']])
try:
    validate_inputs(Image.new('RGB', (MIN_IMAGE_SIDE, int(MIN_IMAGE_SIDE * (MAX_ASPECT_RATIO + 1)))))
except ValueError as exc:
    input_manifest['findings'].append({'input': 'over-wide-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/depth_anything_depth_estimation_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
started = time.perf_counter()
result = pipe.predict(image)
predict_seconds = round(time.perf_counter() - started, 3)
depth = result['depth']
checks = {
    'shape_matches_input': depth.shape == (result['height'], result['width']) == (image.height, image.width),
    'dtype_float32': depth.dtype == np.float32,
    'all_finite': bool(np.isfinite(depth).all()),
    'non_degenerate_range': result['depth_max'] > result['depth_min'],
}
if not all(checks.values()):
    raise RuntimeError(f'depth map failed a sanity check: {checks}')
reference_depth = np.where(probe_record['mask'], probe_record['depth'], 0.0)
report = evaluation_report(result, reference_depth, sample_kind=f"DIODE {probe_record.get('domain', '')} view {probe_record['id']}" if not USE_BYOD else 'BYOD test view')
print({'probe_id': probe_record['id'], 'domain': probe_record.get('domain'), 'seconds': predict_seconds, 'device': pipe.device, 'checks': checks, 'findings': len(input_manifest['findings'])})
print({'verdict': report['verdict'], 'metrics': report['metrics'], 'reason': report['reason']})
assert report['verdict'] == 'sample-sanity' and report['metrics'][0]['id'] == 'abs_rel'

def preview(image, maps):
    tiles = [image.convert('RGB')]
    for m in maps:
        lo, hi = float(m.min()), float(m.max())
        tiles.append(Image.fromarray(np.round((m - lo) / (hi - lo + 1e-9) * 255.0).astype(np.uint8)).convert('RGB'))
    canvas = Image.new('RGB', (image.width * len(tiles), image.height))
    for i, tile in enumerate(tiles):
        canvas.paste(tile, (image.width * i, 0))
    return canvas

try:
    from IPython.display import display
    display(preview(image, [depth]).reduce(2))
except ImportError:
    print({'preview': 'IPython display unavailable; the preview PNG is written in Section 9'})

## 6. The two priors and the zero-shot policy

Three numbers frame the adaptation, all on the 8 test views. The **constant prior** puts every pixel at the image's own median reference depth — an oracle constant, the strongest flat guess. The **vertical-gradient prior** predicts inverse depth rising from the top row to the bottom row and is aligned exactly like a model prediction: what "the ground is nearer" alone buys. The **zero-shot policy** is `pipe.evaluate` on the untouched checkpoint: mean AbsRel and δ1 over views, per domain and per view. The build record saw the constant prior at AbsRel 0.33, the gradient prior at 0.26 and the zero-shot checkpoint at 0.18 (0.08 indoors, 0.28 outdoors) — read the per-domain split; outdoor DIODE views with sky, foliage and 100 m ranges are where a relative-depth model struggles. Two zero-shot depth maps are kept for the before/after comparison in Section 9. About ten seconds on CPU.

In [ ]:
def brief(m):
    return {'abs_rel': round(m['abs_rel'], 4), 'delta1': round(m['delta1'], 4), 'n': m['n']}

priors = prior_baselines(test_records)
print({'constant_prior': brief(priors['constant_prior']), 'baseline': priors['constant_prior']['baseline']})
print({'vertical_gradient_prior': brief(priors['vertical_gradient_prior']), 'baseline': priors['vertical_gradient_prior']['baseline']})
t0 = time.perf_counter()
zero_shot_test = pipe.evaluate(test_records)
print({'zero_shot_policy': zero_shot_test['policy'], 'test': brief(zero_shot_test), 'per_domain': {d: {'abs_rel': round(v['abs_rel'], 4), 'delta1': round(v['delta1'], 4), 'n': v['n']} for d, v in zero_shot_test['per_domain'].items()}, 'verdict': zero_shot_test['verdict'], 'seconds': round(time.perf_counter() - t0, 1)})
print({'per_view': [(r['id'], r['domain'], round(r['abs_rel'], 3), round(r['delta1'], 3)) for r in zero_shot_test['per_image']]})
print({'definitions': zero_shot_test['definitions']})
show = test_records[:2]
zero_shot_maps = {r['id']: pipe.predict(r['image'])['depth'] for r in show}
assert zero_shot_test['abs_rel'] < priors['constant_prior']['abs_rel'] and zero_shot_test['abs_rel'] < priors['vertical_gradient_prior']['abs_rel']

## 7. The policy ladder: neck and head, then a bounded unfreeze, selected against the checkpoint

`pipe.adapt` scores the untouched checkpoint on the validation views first (epoch 0, the zero-shot policy), then trains the DPT neck and head — 2,728,513 parameters — on the frozen backbone for `HEAD_EPOCHS` epochs (the frozen policy), then unfreezes the last `TRAINABLE_BLOCKS` transformer blocks — two by default, 3,550,464 of 24,785,089 parameters; the patch embedding, the position embedding, the earlier blocks and the final norm stay frozen — and trains them with the neck and head for `EPOCHS` more epochs (the unfrozen policy). Every epoch trains one view at a time with AdamW at `LEARNING_RATE` (weight decay 0.01, gradient clipping 1.0, seeded order, no augmentation) under a scale-and-shift-invariant squared error on inverse depth — the prediction is affinely aligned to the reference in closed form before the residual is taken, because the model's scale and shift are free — and is scored on validation by `evaluate`. The epoch with the **lowest validation AbsRel** is kept and its tensors restored, so the selected policy can be the checkpoint itself.

Watch the validation AbsRel: the build record's ladder at 1e-5 went 0.243 (zero-shot) → 0.220 → 0.194 (neck and head) → 0.139 → 0.133 → 0.130 (unfrozen, still falling at epoch 5, which was selected); at 3e-5 the neck-and-head epoch 2 was selected instead, and at 1e-4 the selected unfreeze scored worse than the checkpoint on the held-out split. The validation views are four scans — one view is an eighth of the number.

In [ ]:
HEAD_EPOCHS = 2  # @param {type:"integer"}
EPOCHS = 3  # @param {type:"integer"}
LEARNING_RATE = 1e-5  # @param {type:"number"}
TRAINABLE_BLOCKS = 2  # @param {type:"integer"}

def report_epoch(entry):
    row = {'epoch': entry['epoch'], 'stage': entry['stage'], 'train_loss': round(entry['train_loss'], 4) if entry['train_loss'] is not None else None}
    if entry.get('val'):
        row['val_abs_rel'] = round(entry['val']['abs_rel'], 4)
        row['val_delta1'] = round(entry['val']['delta1'], 4)
    print(row)

t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, trainable_blocks=TRAINABLE_BLOCKS, head_epochs=HEAD_EPOCHS, epochs=EPOCHS, lr=LEARNING_RATE, progress=report_epoch)
adapt_seconds = round(time.perf_counter() - t0, 1)
report_epoch(adapt_result['history'][0])
print({'selected_policy': adapt_result['policy'], 'best_epoch': adapt_result['best_epoch'], 'selection': adapt_result['selection'], 'trainable_neck_head': adapt_result['n_trainable_head'], 'trainable_blocks': adapt_result['n_trainable_blocks'], 'total_parameters': adapt_result['n_total'], 'seconds': adapt_seconds})

## 8. Held-out evaluation

The test split was never used for training or policy selection, and no scan in it appears in the training or validation splits. The selected model is scored exactly as the checkpoint was in Section 6, and the four rows are put side by side: constant prior, vertical-gradient prior, zero-shot policy, selected policy — with per-domain and per-view AbsRel. Read the policy first: if validation kept epoch 0, the last two rows are the same model; otherwise the delta is what the adaptation bought on 8 views. The build record saw AbsRel 0.180 → 0.167 and δ1 0.764 → 0.765 at the default settings — a small gain concentrated outdoors, within the grain of an 8-view split. The cell asserts the selected model beats both priors; it does **not** assert a gain over the checkpoint, because that is the question, not the answer. Eight views from four scans of one seeded split give no dispersion estimate.

In [ ]:
adapted_test = pipe.evaluate(test_records)
adapted_val = pipe.evaluate(val_records)
comparison = {
    metric: {'constant_prior': round(priors['constant_prior'][metric], 4), 'vertical_gradient_prior': round(priors['vertical_gradient_prior'][metric], 4), 'zero_shot': round(zero_shot_test[metric], 4), 'selected_policy': round(adapted_test[metric], 4)}
    for metric in ('abs_rel', 'delta1')
}
comparison['per_domain_abs_rel'] = {d: {'zero_shot': round(zero_shot_test['per_domain'][d]['abs_rel'], 4), 'selected': round(v['abs_rel'], 4)} for d, v in adapted_test['per_domain'].items()}
comparison['per_view_abs_rel'] = {z['id']: {'zero_shot': round(z['abs_rel'], 4), 'selected': round(a['abs_rel'], 4)} for z, a in zip(zero_shot_test['per_image'], adapted_test['per_image'])}
comparison['delta_vs_zero_shot'] = {metric: round(adapted_test[metric] - zero_shot_test[metric], 4) for metric in ('abs_rel', 'delta1')}
comparison['selected_policy'] = adapt_result['policy']
for metric, row in comparison.items():
    print({metric: row})
evaluation_report_payload = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'data_source': data_source,
    'dataset_digests': {name: manifest['digest'] for name, manifest in dataset_manifests.items()},
    'splits': disjoint,
    'scan_summary': summary,
    'single_view_report': report,
    'priors': {k: {kk: vv for kk, vv in v.items() if kk != 'per_image'} for k, v in priors.items()},
    'zero_shot_test': zero_shot_test,
    'validation_metrics': adapted_val,
    'test_metrics': adapted_test,
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/depth_anything_depth_estimation_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report_payload, f, indent=2, ensure_ascii=False)
assert adapted_test['abs_rel'] < priors['constant_prior']['abs_rel'] and adapted_test['abs_rel'] < priors['vertical_gradient_prior']['abs_rel']
print({'report': 'outputs/depth_anything_depth_estimation_evaluation_report.json'})

## 9. Depth maps before and after, export the adapter and reload it

Two test views are predicted by the selected model and rendered beside the zero-shot maps kept in Section 6 (image | zero-shot | selected, normalised per map, brighter = nearer) with each map's aligned AbsRel; `outputs/depth_anything_depth_estimation_preview.png` holds the render. A relative-depth map is a per-image ordering, so read the pair for structure — where the selected model places the ground plane, the walls, the far field — not for absolute brightness.

`pipe.save_artifact` writes the DPT neck and head tensors and, when the unfrozen policy was selected, the trained block tensors — 10.9 MB for neck and head, 25.1 MB with two blocks — as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the base `model.safetensors`, the selected policy, the tensor names, the file size and SHA-256, the training configuration and the epoch history (OUT8). `DepthAnythingPipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest and digest **before** deserialising, refuses any tensor that is not a neck, head or transformer-block tensor of the base, and overlays the tensors onto a freshly loaded base — a new object from files, not the in-memory model (VER2). The cell asserts an identical depth map on a test view and an identical test AbsRel (VER4).

In [ ]:
import shutil

rows = []
canvases = []
for record in show:
    after = pipe.predict(record['image'])['depth']
    before = zero_shot_maps[record['id']]
    rows.append({'id': record['id'], 'domain': record.get('domain', ''), 'zero_shot_abs_rel': round(aligned_abs_rel(before, record['depth'], record['mask']), 4), 'selected_abs_rel': round(aligned_abs_rel(after, record['depth'], record['mask']), 4), 'source_url': record.get('source_url', '')})
    print(rows[-1])
    canvases.append(preview(record['image'], [before, after]))
sheet = Image.new('RGB', (max(c.width for c in canvases), sum(c.height for c in canvases)))
y = 0
for c in canvases:
    sheet.paste(c, (0, y))
    y += c.height
sheet.save('outputs/depth_anything_depth_estimation_preview.png')
try:
    from IPython.display import display
    display(sheet.reduce(3))
except ImportError:
    pass

artifact_dir = Path('outputs/depth_anything_depth_estimation_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'depth_anything_depth_estimation', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'policy': artifact_manifest['adapter']['policy'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...'})

reloaded = DepthAnythingPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
reloaded_map = reloaded.predict(show[0]['image'])['depth']
reloaded_test = reloaded.evaluate(test_records)
parity = {'depth_map_identical': bool(np.array_equal(reloaded_map, pipe.predict(show[0]['image'])['depth'])), 'abs_rel_in_memory': round(adapted_test['abs_rel'], 6), 'abs_rel_reloaded': round(reloaded_test['abs_rel'], 6)}
print({'reload_parity': parity, 'reloaded_policy': reloaded.adapter['policy'], 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert parity['depth_map_identical'] and abs(adapted_test['abs_rel'] - reloaded_test['abs_rel']) < 1e-9

weight_entry = next(entry for entry in MANIFEST['files'] if entry['path'] == WEIGHTS_FILE)
result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': snapshot['files'], 'total_bytes': snapshot.get('total_bytes'), 'fetched_this_run': fetched, 'weight_file': WEIGHTS_FILE, 'weight_format': 'safetensors, digest-verified', 'weight_sha256': weight_entry['sha256']},
    'data_source': data_source,
    'corpus': {'name': CORPUS_NAME, 'release': CORPUS_RELEASE, 'base_url': CORPUS_BASE_URL, 'commit': CORPUS_COMMIT, 'views': len(SAMPLE_RECORDS), 'bytes': CORPUS_BYTES, 'license': CORPUS_LICENSE},
    'inference_contract': {'input_manifest': input_manifest, 'sanity_checks': checks, 'probe_id': probe_record['id'], 'single_view_report': report, 'seconds': predict_seconds},
    'comparison': comparison,
    'before_after': rows,
    'preview_file': 'outputs/depth_anything_depth_estimation_preview.png',
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes'], 'tensors': len(artifact_manifest['tensors']), 'policy': artifact_manifest['adapter']['policy']},
    'reload_parity': parity,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'device': pipe.device, 'dtype': 'float32', 'source': pipe.source},
}
with open('outputs/depth_anything_depth_estimation_result.json', 'w', encoding='utf-8') as handle:
    json.dump(result_payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The untouched Depth Anything V2 Small checkpoint already beats both priors by a wide margin on eight DIODE views (AbsRel 0.18 against 0.26 and 0.33), and a bounded adaptation on 24 training views — neck and head first, then the last two blocks, selected against the checkpoint by validation AbsRel — was selected at its last epoch and lowered held-out AbsRel from 0.180 to 0.167 in the build record, almost all of it outdoors. That is the claim and the finding: the adaptation contract runs a three-policy ladder end to end on a real RGB-D corpus, lets the checkpoint win when it should, and reports the answer against two priors rather than in isolation.

The test split is eight views from four scans of one seeded split of one small corpus with no dispersion estimate — one view is an eighth of every mean, and two views of one scan are nearly the same scene. AbsRel and δ1 are computed after a per-image affine alignment inside 0.6..350 m, so they say how well the *ordering and relative spacing* of depth match the laser reference, not whether any metre is right; the model's output stays relative after adaptation. A gain of 0.013 AbsRel at 1e-5 turned into a loss at 1e-4 in the build sweep, and the neck-and-head-only policy won at 3e-5: the ladder's outcome is a property of this corpus and these hyperparameters, not of the method.

Three things to carry to real data. **Priors first:** the constant and gradient priors on *your* depth maps are the numbers to read before any model's — a scene that a vertical gradient explains is not testing the model. **Leakage:** split by capture session, scan or device (the contract splits by `scan` / `group`, never by view). **Policy:** the untouched checkpoint is a policy too; keep it in the ladder and let validation decide, and keep the learning rate small — a 25 M-parameter model trained on 62 million images has little to learn from 24 views except your sensor's habits.

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can acquire and digest-verify the pinned model snapshot, fetch and digest-verify a real RGB-D corpus, validate the demonstrated dataset contract without leakage, execute the inference contract with a `sample-sanity` metric against a laser reference, run the zero-shot, frozen and unfrozen policies with validation-based selection, evaluate by aligned AbsRel and δ1 against two priors on an independent split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, metric accuracy, a usable acceptance threshold, or production fitness.

**Optional experiments (they do not affect the default path):** set `LEARNING_RATE = 3e-5` and watch the neck-and-head policy win on validation (the build record: epoch 2, held-out 0.175); set `LEARNING_RATE = 1e-4` and read what a hot unfreeze does to the held-out split (selected on validation, 0.186 on test — worse than the checkpoint); set `TRAINABLE_BLOCKS = 4` (the build record: neck and head selected, 0.175); set `EPOCHS = 6` and watch whether validation AbsRel keeps falling; or bring your own RGB-D records through BYOD and read the priors before either policy.

## References

- Repository README: https://github.com/kurtvalcorza/depth-anything-depth-estimation-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/depth-anything-depth-estimation-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/depth-anything-depth-estimation-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/depth-anything/Depth-Anything-V2-Small-hf
- Upstream code: https://github.com/DepthAnything/Depth-Anything-V2
- Depth Anything V2 paper (Yang et al., 2024): https://arxiv.org/abs/2406.09414
- DIODE: A Dense Indoor and Outdoor DEpth Dataset (Vasiljevic et al., 2019; CC BY 4.0): https://diode-dataset.org — https://arxiv.org/abs/1908.00463
- Marigold evaluation mirror serving the pinned DIODE files: https://huggingface.co/datasets/obukhovai/marigold_depth_eval
- Towards Robust Monocular Depth Estimation (MiDaS; the scale-and-shift-invariant loss and aligned evaluation, Ranftl et al., 2020): https://arxiv.org/abs/1907.01341
- Transformers `DepthAnything` documentation: https://huggingface.co/docs/transformers/model_doc/depth_anything
- DIMER Notebook Specification 2.0 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)